<a href="https://colab.research.google.com/github/GinaMCV/S2.01-Nociones-basicas-SQL/blob/main/S8_01_nivel_1_y_2_Anal%C3%ADtica_de_datos_en_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Importacion de librerias

In [249]:
import os
import re
import json
import pathlib
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

# **Nivel 1 - EDA estructurado y limpieza básica**

Tenemos un dataset de empleados obtenido a partir de una encuesta interna. Antes de poder extraer conclusiones, es necesario asegurarnos de que los datos son correctos, consistentes y analizables.

Ejercicio 1: Exploración inicial del dataset

🎯 Objetivo: entender la estructura general del dataset y detectar posibles problemas antes de empezar a limpiar o transformar datos.
En este ejercicio debes realizar una exploración inicial del DataFrame para responder, como mínimo, a las siguientes preguntas:

- ¿Cuántas filas y columnas tiene el dataset?
- ¿Qué tipo de datos contiene cada columna?
- ¿Hay filas duplicadas?
- ¿Los datos parecen coherentes a primera vista?
- ¿Detectas posibles problemas o inconsistencias evidentes?

Para ello deberás:

- » Analizar las dimensiones del dataset
- » Mostrar una muestra representativa de filas
- » Revisar los tipos de variables
- » ¿Los datos parecen coherentes a primera vista?
- » Detectar filas duplicadas
- » Identificar y describir posibles problemas iniciales (sin solucionarlos todavía)


In [250]:
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/phyton_especialidad/sprint8_complex.xlsx")


- las 3 primeras filas que son titulo, subtitulo y una fila vacia no contienen informacion que concuerde con la de las columnas
- La fila que contiene la informacion del titulo se encuentra en la fila 3
- la primera columna (Unnamed: 0) es un índice guardado como columna.
- El nombre de la columna de cargo también llega corrupto (CÃ rrec).

Se debe leer bien la cabecera, eliminar o reindexar la columna índice y reparar los nombres de columna.


In [251]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/phyton_especialidad/sprint8_complex.xlsx", header=3)

- ¿Cuántas filas y columnas tiene el dataset?
  - La base de datos esta compuesta por 1007 filas y 20 columnas


In [252]:
filas = df.shape[0]
columnas = df.shape[1]
print(f"El dataset tiene {filas} filas y {columnas} columnas")

El dataset tiene 1007 filas y 20 columnas


- Muestra representativa de filas
  - En la muestra ya se ven tildes y simbolos corruptos (`Ã `, `â‚¬`)
  - Salarios escritos con formatos distintos
  - Muchos huecos en las columnas de coche
  - Filas que no son empleados reales

Se debe abordar la codificación de texto y eliminación de filas basura


In [253]:
print("PRIMERAS 6 FILAS")
display(df.head(6))

print("\nÚLTIMAS 6 FILAS")
display(df.tail(6))

print("\n6 FILAS ALEATORIAS")
display(df.sample(6, random_state=42))

PRIMERAS 6 FILAS


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
5,5,Alexia,Puig,25014631B,Espanya,Las Palmas de Gran Canaria,1.0,8.0,1978.0,D,"2,069 â‚¬",NaN,True,Grup B,Cap de projecte,0.0,NaN,NaN,NaN,19.9



ÚLTIMAS 6 FILAS


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1001,651,Kasper,Pettersen,62042563B,Noruega,Oslo,11.0,5.0,1996.0,D,2.290 â‚¬,NaN,True,Grup B,Senior analyst,0.0,True,80000.0,0.5,6.5
1002,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1006,1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,molts,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN



6 FILAS ALEATORIAS


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
927,927,Anna,PÃ©rez Serra,81260169S,Espanya,Barcelona,11.0,2.0,1983.0,H,1.927 â‚¬,True,NaN,Grup B,Senior analyst,3.0,NaN,NaN,NaN,16.6
630,630,VÃ­ctor,DomÃ¨nech,22730042T,Espanya,Zaragoza,7.0,3.0,1995.0,H,"1,579 â‚¬",NaN,True,Grup B,TÃ¨cnic IT,0.0,NaN,NaN,NaN,11.8
682,682,Carmen,DÃ­az Casals,89650979N,Espanya,Murcia,11.0,10.0,1968.0,H,"2,425 â‚¬",NaN,True,Grup C,Analista junior,0.0,NaN,NaN,NaN,21.1
514,514,VÃ­ctor,Roca MuÃ±oz,25081699D,Espanya,Palma,8.0,2.0,1976.0,D,943â‚¬,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,17.3
365,365,LucÃ­a,DomÃ¨nech,84449447A,Espanya,Zaragoza,1.0,6.0,1962.0,D,966 â‚¬,NaN,True,Grup A,Data Analyst,0.0,True,72237.0,10.8,15.4
655,655,VÃ­ctor,Serra,91229370H,Espanya,Palma,5.0,3.0,1968.0,D,"1,557 â‚¬",NaN,True,Grup B,Analista,NaN,True,29434.0,0.5,15.7


- ¿Qué tipo de datos contiene cada columna?
  - Salari mensual es texto en lugar de numero
  - las fechas son float porque tienen nulos
  - Fills, No Fills y Te_cotxe son object
  - se identificaron nulos, sobre todo en Km_anuals (69 %), Consum (74 %), Te_cotxe (35 %) y Nombre_fills (30 %)

Se debe convertir tipos (salario a número, fechas a datetime, booleanos) y decidir una estrategia para los nulos.


In [254]:
print("\nTipo de datos por columna")
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "únicos": df.nunique(),
}))


Tipo de datos por columna


,dtype,nulos,% nulos,únicos
Unnamed: 0,int64,0,0.00,1007
Nom,object,1,0.10,103
Cognoms,object,1,0.10,468
DNI,object,1,0.10,991
País d'origen,object,1,0.10,6
Ciutat,object,1,0.10,45
Dia de Naixement,float64,1,0.10,35
Mes de Naixement,float64,1,0.10,15
Any de Naixement,float64,1,0.10,51
Gènere,object,41,4.07,9


- ¿Hay filas duplicadas?
  - si incluimos la columna indice no hay duplicados porque es unica y los oculta, al excluirla aparecen 4 filas duplicadas exactas
  
Se deben eliminar las copias exactas



In [255]:
duplicados = df.duplicated().sum()
print(f"Duplicados con todas las columnas: {duplicados} " )

cols_datos = df.columns[1:]
duplicados_sin_indice = df.duplicated(subset=cols_datos).sum()
print(f"Duplicados ignorando la primera columna: {duplicados_sin_indice} ")

display(df[df.duplicated(subset=cols_datos, keep=False)].sort_values("DNI"))

Duplicados con todas las columnas: 0 
Duplicados ignorando la primera columna: 4 


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
36,464,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6
40,1001,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6
730,730,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
372,372,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1002,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7



- Hay DNIs repetidos en filas que no son copias exactas (mismo DNI con ciudades o salarios distintos)

Se debe decidir qué hacer con los DNI conflictivos, revisar si pertenenen a la misma persona o son errores de ingreso.


In [256]:
cols_datos = df.columns[1:]
dni_rep = df[df["DNI"].duplicated(keep=False)].drop(df[df.duplicated(subset=cols_datos, keep=False)].index)
filas = len(dni_rep)
cant_afectado = dni_rep["DNI"].nunique()
print(f"EN {filas} filas hay {cant_afectado} DNIs afectados")

display(dni_rep.sort_values("DNI")[[df.columns[0], "Nom", "Cognoms", "DNI", "Ciutat", "Salari mensual"]])

EN 16 filas hay 5 DNIs afectados


,Unnamed: 0,Nom,Cognoms,DNI,Ciutat,Salari mensual
250,250,Laia,Navarro Vila,15909601K,Bilbao,1235 euros
604,604,Laia,Navarro,15909601K,Bilbao,"1,676 â‚¬"
654,654,Oriol,GÃ³mez,16618423V,Madrid,1731 euros
754,754,Oriol,GÃ³mez,16618423V,Barcelona,2.358 â‚¬
758,758,Oriol,GÃ³mez,16618423V,MÃ laga,1879 euros
759,759,Oriol,GÃ³mez,16618423V,Sevilla,"2,000 â‚¬"
558,558,Alexia,PÃ©rez DÃ­az,49529438G,MÃ laga,"1,176 â‚¬"
692,692,Alexia,PÃ©rez DÃ­az,49529438G,MÃ laga,1663â‚¬
89,89,Guillem,Pujol Costa,54013571W,A CoruÃ±a,1.574 â‚¬
142,142,Marta,Ruiz,54013571W,Palma,"1,276 â‚¬"


- ¿Los datos parecen coherentes a primera vista?
  - las categoricas mezclan codigos y variantes (genero D/Dona/F/H/Home/M/A/NC/??, Cargos Data Analyst/Data analyst, Cap de projecte/Cap de Projecte, Admin./Administratiu…)
  - el nombre de la columna CÃ rrec llega corrupto

Se deben unificar las categorias y nombres de columnas



In [257]:
col_carrec = df.columns[14]
for columna in ["País d'origen", "Gènere", "Grup Professional", col_carrec, "Fills", "No Fills", "Te_cotxe"]:
    print("\n"+ df[columna].value_counts(dropna=False).to_string())


País d'origen
Espanya     719
ItÃ lia     108
FranÃ§a      86
Alemanya     48
Noruega      44
NaN           1
Narnia        1

Gènere
D       325
H       280
Dona    120
F       107
Home     52
NaN      41
M        34
A        24
NC       23
??        1

Grup Professional
Grup A    466
Grup B    357
Grup C    140
Grup D     41
NaN         2
Z           1

CÃ rrec
Data Analyst             142
TÃ¨cnic IT               129
Responsable de vendes    125
Senior analyst           122
Analista                 116
Analista junior          107
Administratiu            105
Cap de projecte           94
Project lead              18
Data analyst              18
Cap de Projecte           13
Analyst                    9
Admin.                     6
NaN                        2
???                        1

Fills
NaN     588
True    419

No Fills
True     584
NaN      422
False      1

Te_cotxe
NaN      349
False    330
True     328



- En las columnas numéricas, los minimos y maximos delatan valores imposibles (dia −99, año 3000, km negativos o de 1.000.000)

Se deben tratar los valores fuera de rango


In [258]:
num = ["Dia de Naixement", "Mes de Naixement", "Any de Naixement", "Nombre_fills", "Km_anuals", "Consum_mitja_L_100km", "Temperatura_mitjana_ciutat"]
display(df[num].describe().round(2))

,Dia de Naixement,Mes de Naixement,Any de Naixement,Nombre_fills,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
count,1006.00,1006.00,1006.00,709.00,315.00,261.00,936.00
mean,15.03,8.56,2002.23,1.20,55951.05,11.02,15.38
std,10.33,14.30,143.12,1.39,62095.44,8.78,7.28
min,-99.00,0.00,1940.00,0.00,-50.00,0.50,-10.00
25%,7.00,3.25,1969.00,0.00,22632.00,4.70,12.80
50%,15.00,6.00,1982.00,1.00,56135.00,9.80,15.50
75%,24.00,10.00,1995.00,2.00,80000.00,13.60,18.30
max,32.00,99.00,3000.00,5.00,1000000.00,25.00,42.00


- ¿Detectas posibles problemas o inconsistencias evidentes?
  1. Estructura, filas vacías y filas de prueba
  2. Codificación de texto corrupta (mojibake)
  3. Fechas de nacimiento imposibles
  4. Salario: formatos mezclados y texto
  5. Género y cargo: categorías inconsistentes
  6. Fills / No Fills / Nombre_fills: columnas contradictorias
  7. Coche, kilómetros, consumo y temperatura: valores sospechosos y nulos
  8. Formato del DNI




 1. Estructura, filas vacías y filas de prueba
    - Hay una fila totalmente vacia y una fila de prueba (`???`, `XXXX`, `Narnia`, grupo `Z`)
    - En 44 filas el indice no coincide con su posicion

Se deben eliminar y revisar el orden

In [259]:
cant_vacias = df[df.columns[1:]].isna().all(axis=1).sum()
print(f"Filas totalmente vacias sin incluir indice: {cant_vacias} " )
display(df.loc[df[df.columns[1:]].isna().all(axis=1)])

filas_filtradas = df[(df["Nom"] == "???") & (df["País d'origen"] == "Narnia")]
prueba = len(filas_filtradas)
print(f"\nFilas con valores de prueba : {prueba}")
display(filas_filtradas)

sin_orden = (df.iloc[:, 0] != range(len(df))).sum()
print(f"\nFilas donde el índice no sigue el orden 0,1,2,...: {sin_orden} " )
display(df.loc[(df.iloc[:, 0] != range(len(df)))].head(4))

Filas totalmente vacias sin incluir indice: 1 


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Filas con valores de prueba : 1


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1006,1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,molts,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN



Filas donde el índice no sigue el orden 0,1,2,...: 44 


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
8,540,Nora,Andersen Pedersen,37289414J,Noruega,Stavanger,8.0,3.0,1963.0,D,"1,338 â‚¬",NaN,True,Grup B,Analista junior,0.0,False,NaN,NaN,7.6
17,375,Sigrid,Olsen,70604045C,Noruega,Bergen,19.0,6.0,1964.0,D,"1,385 â‚¬",True,True,Grup A,Admin.,0.0,True,80000.0,25.0,8.8
36,464,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6
40,1001,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6


2. Codificación de texto corrupta (mojibake)
  - El texto viene con la codificacion rota (UTF-8 leido como latin-1) en salarios, apellidos, ciudades y paises
Se debe reparar antes de agrupar o comparar texto ya que Málaga y MÃ laga contaran como categorias distintas


In [260]:
for columna in ["Nom", "Cognoms", "Ciutat", "Salari mensual"]:
    cantidad = df[columna].dropna().astype(str).str.contains("Ã|â‚").sum()
    print(f"{columna:<16} filas con caracteres corruptos: {cantidad}")

print("\nEjemplos:", df["Cognoms"].dropna().loc[lambda texto: texto.str.contains("Ã|â‚")].head(4).tolist(),
      df["Salari mensual"].head(2).tolist())

Nom              filas con caracteres corruptos: 128
Cognoms          filas con caracteres corruptos: 376
Ciutat           filas con caracteres corruptos: 160
Salari mensual   filas con caracteres corruptos: 842

Ejemplos: ['MuÃ±oz', 'GÃ³mez Vila', 'LÃ³pez', 'GarcÃ\xada'] ['1,469 â‚¬', '2,718 â‚¬']


3. Fechas de nacimiento imposibles
  - En 78 filas dia, mes y año:
    - Son invalidos a la vez (−99/−5/0/32, 0/13/99 y 1940/2019/2026/3000)
    - Parecen valores centinela y no fechas reales
    - Los años normales van de 1958 a 2004

Se deben convertir a nulos, construir una fecha valida y calcular la edad


In [261]:
cant_dia = (~df["Dia de Naixement"].between(1, 31) & df["Dia de Naixement"].notna()).sum()
dia = sorted(df.loc[~df["Dia de Naixement"].between(1, 31), "Dia de Naixement"].dropna().unique().tolist())
print(f"Dias fuera de 1-31: {cant_dia}, dias en el dataset {dia} ")

cant_mes = (~df["Mes de Naixement"].between(1, 12) & df["Mes de Naixement"].notna()).sum()
mes = sorted(df.loc[~df["Mes de Naixement"].between(1, 12), "Mes de Naixement"].dropna().unique().tolist())
print(f"Meses fuera de 1-12: {cant_mes}, meses en el dataset {mes} ")

anios = sorted(df["Any de Naixement"].dropna().unique())
cant_anios = set(anios)
anormales = sorted(anio for anio in anios if (anio - 1 not in cant_anios) and (anio + 1 not in cant_anios))
normales = [anio for anio in anios if anio not in anormales]

print(f"\nRango de años anormales: {min(anormales):.0f} – {max(anormales):.0f} (con huecos)")
print(f"Rango de años normales: {min(normales):.0f} – {max(normales):.0f} (sin huecos)")

print("\nAños con mas frecuencia de anomalia:")
print(df["Any de Naixement"].value_counts().loc[anormales])


Dias fuera de 1-31: 78, dias en el dataset [-99.0, -5.0, 0.0, 32.0] 
Meses fuera de 1-12: 78, meses en el dataset [0.0, 13.0, 99.0] 

Rango de años anormales: 1940 – 3000 (con huecos)
Rango de años normales: 1958 – 2004 (sin huecos)

Años con mas frecuencia de anomalia:
Any de Naixement
1940.0    17
2019.0    23
2026.0    18
3000.0    20
Name: count, dtype: int64


4. Salario: formatos mezclados y texto
  - El salario mezcla 6 formatos (1,469 €, 1.789 €, 786 €, 786€, 1121 euros) y 37 valores en texto o sin dato (n/d, dos mil, molts…)
  - La coma y el punto se usan como separador de miles

se debe normalizar a un numero, con cuidado de no confundir el separador de miles con un decimal


In [262]:
sal = df["Salari mensual"].dropna().astype(str)
formato = pd.Series("texto / otro", index=sal.index)
formato[sal.str.match(r"^\d+ euros$")]        = "numero + palabra 'euros' ejemplo: 1121 euros"
formato[sal.str.match(r"^\d+[^\d\s]+$")]      = "numero simple con € pegado ejemplo: 786€"
formato[sal.str.match(r"^\d+ .{1,3}$")]       = "numero simple con € pegado ejemplo: 786 €"
formato[sal.str.match(r"^\d{1,3}\.\d{3} ")]   = "numero con punto de miles ejemplo: 1.789 € "
formato[sal.str.match(r"^\d{1,3},\d{3} ")]    = "numero con coma de miles ejemplor: 1,469 €"
print(formato.value_counts())
print("\nValores en texto:", sal[formato == "texto / otro"].value_counts().to_dict())

numero con coma de miles ejemplor: 1,469 €      499
numero simple con € pegado ejemplo: 786€        171
numero + palabra 'euros' ejemplo: 1121 euros    127
numero simple con € pegado ejemplo: 786 €       112
numero con punto de miles ejemplo: 1.789 €       60
texto / otro                                     37
Name: count, dtype: int64

Valores en texto: {'n/d': 14, 'mil vuit-cents': 9, 'dos mil': 7, 'tres mil': 6, 'molts': 1}


5. Género y cargo: categorías inconsistentes
  - Para el mismo concepto hay varios codigos y no queda claro que significan M y A ademas hay 41 vacios y un ???
  - Los cargos tienen variantes de mayusculas, idioma y abreviaturas
  
Se debe definir un mapeo de categorías y decidir que hacer con los valores ambiguos.


In [263]:
generos = df["Gènere"].value_counts(dropna=False).to_dict()
print(f"Generos: {generos} " )

col_carrec = df.columns[14]
cargos = df[col_carrec].value_counts(dropna=False).to_string()
print(f"\n{cargos}")


Generos: {'D': 325, 'H': 280, 'Dona': 120, 'F': 107, 'Home': 52, nan: 41, 'M': 34, 'A': 24, 'NC': 23, '??': 1} 

CÃ rrec
Data Analyst             142
TÃ¨cnic IT               129
Responsable de vendes    125
Senior analyst           122
Analista                 116
Analista junior          107
Administratiu            105
Cap de projecte           94
Project lead              18
Data analyst              18
Cap de Projecte           13
Analyst                    9
Admin.                     6
NaN                        2
???                        1


6. Fills / No Fills / Nombre_fills: columnas contradictorias
 - Fills y No Fills son redundantes y se contradicen entre si y con Nombre_fills:
   - 49 con ambas marcadas
   - 52 sin ninguna
   - 34 con Fills pero 0 hijos
   - 20 con No Fills pero hijos
   
Se debe decidir cual es la fuente de verdad (probablemente Nombre_fills) y resolver los conflictos.


In [264]:
ambos_informados = (df["Fills"].notna() & df["No Fills"].notna()).sum()
print(f"Fills y No Fills informados a la vez: {ambos_informados}")

ninguno_informado = (~df["Fills"].notna() & ~df["No Fills"].notna()).sum()
print(f"\nNi Fills ni No Fills: {ninguno_informado} ")

fills_con_conteo_cero = (df["Fills"].notna() & (df["Nombre_fills"] == 0)).sum()
print(f"\nFills=True pero Nombre_fills = 0: {fills_con_conteo_cero} ")

nofills_con_conteo_positivo = ((df["No Fills"] == True) & (df["Nombre_fills"] > 0)).sum()
print(f"\nNo Fills=True pero Nombre_fills > 0: {nofills_con_conteo_positivo}")

resumen_valores_nofills = df["No Fills"].value_counts(dropna=False).to_dict()
print(f"\nValores de No Fills:{resumen_valores_nofills}" )

Fills y No Fills informados a la vez: 49

Ni Fills ni No Fills: 52 

Fills=True pero Nombre_fills = 0: 34 

No Fills=True pero Nombre_fills > 0: 20

Valores de No Fills:{True: 584, nan: 422, False: 1}


7. Coche, kilómetros, consumo y temperatura: valores sospechosos y nulos

Hay valores sospechosos en:
- Km_anuals (negativos, 1.000.000, 74 filas en exactamente 80.000)
- Consum (119 filas en 0,5 o 25,0), tienen coche sin consumo (67)
- Temperatura (66 filas en −10, −5, 38 o 42 °C)

Se debe detectar outliers y decidir entre eliminarlos, marcarlos como nulos o imputarlos, la temperatura se podria reconstruir por ciudad

Kilometros

In [265]:
kl_negartivo = (df["Km_anuals"] < 0).sum()
print(f"Km negativos: {kl_negartivo}")

kl_valor_alto = (df["Km_anuals"] >= 1e6).sum()
print(f"Km = 1.000.000: {kl_valor_alto}")

kl_repetidos = df["Km_anuals"].value_counts().head(4).to_dict()
print(f"Km más repetidos: {kl_repetidos} ")

Km negativos: 6
Km = 1.000.000: 1
Km más repetidos: {80000.0: 74, 20000.0: 21, 65950.0: 20, 50000.0: 11} 


Consumo

In [266]:
consumo = df["Consum_mitja_L_100km"].isin([df["Consum_mitja_L_100km"].min(), df["Consum_mitja_L_100km"].max()]).sum()
print(f"Consumo exactamente 0.5 ó 25.0: {consumo}")

Te_cotxe_consumo = ((df["Te_cotxe"] == True) & df["Consum_mitja_L_100km"].isna()).sum()
print(f"Te_cotxe=True sin consumo: {Te_cotxe_consumo}")

Consumo exactamente 0.5 ó 25.0: 119
Te_cotxe=True sin consumo: 67


Temperatura

In [267]:
temps = pd.Series(sorted(df["Temperatura_mitjana_ciutat"].dropna().unique()))

gap_prev = temps.diff().fillna(np.inf)
gap_next = (-temps.diff(-1)).fillna(np.inf)

tem_anormales = temps[(gap_prev > 1) & (gap_next > 1)]

temp_alta = tem_anormales.nlargest(2)
temp_baja = tem_anormales.nsmallest(2)

print(f"Temperaturas más ALTAS: {temp_alta.tolist()}")
print(f"Temperaturas más BAJAS: {temp_baja.tolist()}")

valores = pd.concat([temp_alta, temp_baja])
n_filas = df["Temperatura_mitjana_ciutat"].isin(valores).sum()
temperaturas = ', '.join(map(str, valores))
print(f"Filas con esas temperaturas ({temperaturas}): Cantidad: {n_filas}")


Temperaturas más ALTAS: [42.0, 38.0]
Temperaturas más BAJAS: [-10.0, -5.0]
Filas con esas temperaturas (42.0, 38.0, -10.0, -5.0): Cantidad: 66


8. Formato del DNI
  - El formato es correcto salvo XXXX, pero casi ninguna letra pasa el algoritmo oficial (los datos estan "generados aleatoriamente")
  - Personas de otros paises tienen DNI con formato español

en este caso no usar la letra del DNI como criterio de validez; el DNI sirve solo como identificador para detectar duplicados

In [268]:
dni = df["DNI"].dropna().astype(str)
formato = dni.str.match(r"^\d{8}[A-Z]$")
LETRAS = "TRWAGMYFPDXBNJZSQVHLCKE"
formato_valido = formato == True
formato_invalido = ~formato_valido
ok = dni[formato_valido].map(lambda letra: LETRAS[int(letra[:8]) % 23] == letra[8])
correctos = ok.sum()
letra_incorrecta = (~ok).sum()
invalidos_total = formato_invalido.sum()

print(f"DNI con formato 8 dígitos + letra: {formato.sum()} de {len(dni)}")
print(f"Letra del DNI correcta: {correctos}")
print(f"Letra del DNI incorrecta (pero formato válido): {letra_incorrecta}")
print(f"No cumplen el formato, {dni[~formato].tolist()}: {invalidos_total} ")

print("\nPaíses (todos tienen DNI de formato español):", df["País d'origen"].value_counts().to_dict())

DNI con formato 8 dígitos + letra: 1005 de 1006
Letra del DNI correcta: 46
Letra del DNI incorrecta (pero formato válido): 959
No cumplen el formato, ['XXXX']: 1 

Países (todos tienen DNI de formato español): {'Espanya': 719, 'ItÃ\xa0lia': 108, 'FranÃ§a': 86, 'Alemanya': 48, 'Noruega': 44, 'Narnia': 1}


# Ejercicio 2: Validaciones básicas de los datos y decisiones de eliminación
🎯 Objetivo: asegurar que los datos cumplen unas condiciones mínimas de calidad y determinar qué filas pueden seguir siendo analizadas y cuáles deben descartarse.

En este ejercicio tendrás que validar aspectos clave del dataset que son críticos en cualquier contexto real de análisis de datos, tomando decisiones justificadas sobre la validez o eliminación de registros .


Aspectos a validar:

- » ¿Existen identificadores únicos (DNI) para cada persona?
- » ¿Qué vas a hacer con las filas duplicadas o registros repetidos?
- » ¿Existen valores obligatorios que faltan o información incoherente?
- » ¿Detectos formatos incorrectos , errores de codificación o valores imposibles?

Concretamente, tendrás que:

- » Corregir posibles problemas de codificación y acentuación en los nombres de las columnas y en los valores de texto, si procede.
- » Detectar y eliminar filas duplicadas exactas, indicando cuántos registros se han eliminado.
- » Comprobar la unicidad del DNI. En caso de detectar DNIs duplicados, deberás analizar la coherencia de la información asociada y descartar o anular los registros que no sean fiables, justificando claramente la decisión tomada.
- » Identificar los campos clave que no deberían contener valores nulos, detectar registros con información insuficiente y eliminar únicamente aquellas filas completamente vacías o con información irrecuperable o no sustituible, explicando por qué no es posible ni razonable imputarlas.


In [269]:
df_raw = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/phyton_especialidad/sprint8_complex.xlsx", header=3)
df = df_raw.copy()

Lista de registro y descartados

In [270]:
registro = []
descartados = []

def descarta(mascara, motivo):
    """Saca del data frame las filas marcadas y las guarda en la lista descartados con el motivo."""
    global df
    quitadas = df[mascara].assign(motivo=motivo)
    descartados.append(quitadas)
    registro.append((motivo, len(quitadas)))
    df = df[~mascara]
    return len(quitadas)

## 1. Codificación y acentuación (nombres de columnas y valores de texto)


**Qué observo:** el problema era de codificación (UTF-8 leído como latin-1). Afectaba al nombre de la columna `Càrrec` y a los valores de `Nom`, `Cognoms`, `País d'origen`, `Ciutat`, `Càrrec` y `Salari mensual` (donde `€` se veía como `â‚¬`).

**Decisión:** se repara el texto, **no se elimina ninguna fila**. Se hace **primero** porque, si no, `Muñoz` y `MuÃ±oz` contarían como valores distintos y ocultarían duplicados.



In [271]:
# Restos típicos de UTF-8 leido como latin-1:  'Ã', 'â‚' (de '€'), 'Â'
MOJIBAKE = "\u00c3|\u00e2\u201a|\u00c2"

def repara(texto):
    """Si el texto tiene restos de mala codificación, lo reinterpreta como UTF-8."""
    if not isinstance(texto, str) or not re.search(MOJIBAKE, texto):
        return texto
    for codificacion in ("cp1252", "latin-1"):
        try:
            return texto.encode(codificacion).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return texto


df.columns = [repara(c) for c in df.columns]
df = df.rename(columns={"Unnamed: 0": "idx_origen"})

cols_texto = df.select_dtypes(include=["object", "string"]).columns
for c in cols_texto:
    df[c] = df[c].map(repara)

cambios = {c: int((df_raw.iloc[:, i].fillna("").astype(str) != df[c].fillna("").astype(str)).sum())
           for i, c in enumerate(df.columns) if c in cols_texto}

print("\nCeldas reparadas por columna:")
print(pd.Series(cambios)[lambda s: s > 0].to_string())

restos = sum(df[c].dropna().astype(str).str.contains(MOJIBAKE).sum() for c in cols_texto)
print(f"\nCaracteres corruptos que quedan: {restos}")

print()
filas = df_raw.index[df_raw["Cognoms"].fillna("").str.contains(MOJIBAKE)][:4]
display(pd.DataFrame({"Apellidos antes": df_raw.loc[filas, "Cognoms"], "Apellidos después": df.loc[filas, "Cognoms"],
                      "Salario antes": df_raw.loc[filas, "Salari mensual"], "Salario después": df.loc[filas, "Salari mensual"]}))


Celdas reparadas por columna:
Nom               128
Cognoms           376
País d'origen     194
Ciutat            160
Salari mensual    842
Càrrec            129

Caracteres corruptos que quedan: 0



,Apellidos antes,Apellidos después,Salario antes,Salario después
1,MuÃ±oz,Muñoz,"2,718 â‚¬","2,718 €"
6,GÃ³mez Vila,Gómez Vila,"2,349 â‚¬","2,349 €"
7,LÃ³pez,López,"2,855 â‚¬","2,855 €"
9,GarcÃ­a,García,"1,450 â‚¬","1,450 €"


## 2. Filas completamente vacías

**Qué observo:** hay una fila sin ningún valor (aparte del índice).

**Decisión:** se elimina. No aporta ninguna información y no hay nada a partir de lo cual reconstruirla.

In [272]:
cols_dato = [c for c in df.columns if c != "idx_origen"]

vacias = df[cols_dato].isna().all(axis=1)
indice = df.loc[vacias, "idx_origen"].tolist()

descarta(vacias, "Fila completamente vacia")
print(f"Filas completamente vacias eliminadas: { vacias.sum()} → idx_origen {indice}")
print(f"Dimensiones: {df.shape}")

Filas completamente vacias eliminadas: 1 → idx_origen [1005]
Dimensiones: (1006, 20)


## 3. Filas duplicadas exactas

**Qué observo:** 4 parejas de filas idénticas en todas las columnas (Oriol Ferrer, Ida Karlsen, Claudia Gil, Alba Muñoz). Las copias están al final del archivo.

**Decisión:** se conserva la primera aparición y se eliminan **4 registros**. Son el mismo registro repetido, no hay información nueva ni contradictoria.

In [273]:
filas_dup = df.duplicated(subset=cols_dato, keep=False)
print(f"Filas implicadas en copias exactas: {filas_dup.sum()}" )

duplicadas = df.duplicated(subset=cols_dato, keep="first")

descarta(duplicadas, "Duplicado exacto (se conserva la 1ª aparición)")

print(f"\nRegistros duplicados exactos eliminados: {duplicadas.sum()}")

print(f"\nDimensiones: {df.shape}")

Filas implicadas en copias exactas: 8

Registros duplicados exactos eliminados: 4

Dimensiones: (1002, 20)


## 4. Identificador: formato y unicidad del DNI

### 4.1 Formato del DNI

**Qué observo:** una fila tiene `XXXX` como DNI, y el resto de sus campos también son de prueba (`???`, `###`, país `Narnia`, ciudad `Desconeguda`, salario `molts`, grupo `Z`, día −99, mes 99, año 3000).

**Decisión:** se elimina. No es un empleado real: todos los campos son valores de relleno, por lo que la información es irrecuperable y no sustituible

In [274]:
formato_dni = df["DNI"].astype(str).str.fullmatch(r"\d{8}[A-Z]")

descarta(~formato_dni, "DNI inexistente: fila de prueba (valores de relleno)")

print(f"DNI con formato incorrecto eliminado: {(~formato_dni).sum()}")

print(f"Dimensiones: {df.shape}")

DNI con formato incorrecto eliminado: 1
Dimensiones: (1001, 20)


### 4.2 Unicidad del DNI

**Qué observo:** una vez eliminadas las copias exactas, 5 DNI aparecen en 16 filas y **ninguno** es una simple repetición; hay dos patrones:

**A) DNI compartido por personas distintas** (`54013571W`: Guillem Pujol y Marta Ruiz; `90585882A`: Sara Serra ×3, Oriol Vila, Pol Serra y Xavier Vidal). Un DNI identifica **a una sola persona**, así que como mínimo todos menos uno son erróneos, y no hay ningún campo que permita saber cuál es el correcto.

**B) Misma persona con datos contradictorios** (`15909601K` Laia Navarro, `16618423V` Oriol Gómez ×4 —en 4 ciudades distintas—, `49529438G` Alexia Pérez). Coinciden el nombre y la fecha de nacimiento, pero difieren en **salario, grupo profesional, cargo** (y a veces ciudad).

**Decisión: se descartan todas las filas de estos DNI (16 registros).**
- **Caso A:** no se puede determinar qué registro es el bueno; conservar uno sería una apuesta arbitraria.
- **Caso B:** no hay fecha/hora de la respuesta ni ningún otro criterio para saber qué versión es la vigente.
- **No se anula solo el DNI** (dejando la fila): el DNI es un campo clave no sustituible, y una fila sin identificador no se puede verificar ni cruzar con nada.
- **Coste:** 16 filas = 1,6 % del dataset; el impacto en el análisis es mínimo y evitamos introducir datos falsos.
- Todo queda registrado en `descartados`, de manera que la decisión se puede revisar.

In [275]:
rep = df[df["DNI"].duplicated(keep=False)]
print(f"Filas con DNI repetido: {len(rep)}  |  DNI afectados: {rep['DNI'].nunique()}")

campos_persona   = ["Nom", "Dia de Naixement", "Mes de Naixement", "Any de Naixement"]
campos_comparar  = [c for c in cols_dato if c != "DNI"]

analisis = []
for dni, g in rep.groupby("DNI"):
    personas = g[campos_persona].drop_duplicates().shape[0]
    analisis.append({
        "DNI": dni,
        "filas": len(g),
        "personas distintas": personas,
        "tipo": "DNI compartido por personas distintas" if personas > 1 else "Misma persona, datos contradictorios",
        "campos que discrepan": ", ".join(c for c in campos_comparar if g[c].nunique(dropna=False) > 1),
    })
display(pd.DataFrame(analisis))

dni_duplicados = df["DNI"].duplicated(keep=False)

descarta(dni_duplicados, "DNI duplicado con información contradictoria/no fiable")

print(f"\nRegistros descartados por DNI no fiable: {dni_duplicados.sum()}")

print(f"\nDimensiones: {df.shape}")

Filas con DNI repetido: 16  |  DNI afectados: 5


,DNI,filas,personas distintas,tipo,campos que discrepan
0,15909601K,2,1,"Misma persona, datos contradictorios","Cognoms, Salari mensual, No Fills, Grup Profes..."
1,16618423V,4,1,"Misma persona, datos contradictorios","Ciutat, Gènere, Salari mensual, Fills, Grup Pr..."
2,49529438G,2,1,"Misma persona, datos contradictorios","Salari mensual, Grup Professional, Càrrec, Nom..."
3,54013571W,2,2,DNI compartido por personas distintas,"Nom, Cognoms, Ciutat, Dia de Naixement, Mes de..."
4,90585882A,6,4,DNI compartido por personas distintas,"Nom, Cognoms, Ciutat, Dia de Naixement, Mes de..."



Registros descartados por DNI no fiable: 16

Dimensiones: (985, 20)


## 5. Campos clave y valores obligatorios que faltan

**Qué observo:**
- **Identificadores** (`DNI`, `Nom`, `Cognoms`) y **campos obligatorios** ya no tienen ningún nulo tras las bajas anteriores: los únicos nulos venían de la fila vacía y de una fila del DNI con datos incoherentes.
- Hay muchos nulos en los **campos opcionales/condicionales**: `Km_anuals`, `Consum`, `Te_cotxe`, `Nombre_fills`, `Fills`, `No Fills`, `Temperatura` y `Gènere`. En muchos casos no es un dato perdido sino consecuencia del diseño (quien no tiene coche no tiene consumo; `Fills`/`No Fills` solo llevan `True`).
- Hay **nulos disfrazados** de texto: salarios `n/d` (y algunos escritos con letras).

**Decisión: no se elimina ninguna fila más.** Solo eliminamos filas vacías o con información irrecuperable, y aquí ninguna lo es.
- **Por qué no imputar `DNI`, `Nom`, `Cognoms`:** son identidad; inventarlos crearía personas ficticias y rompería la unicidad. Por eso ninguna fila sin identificador se puede conservar (ahora hay 0).
- **Por qué no se eliminan las filas con campos opcionales nulos:** la fila conserva identidad, perfil laboral y demográfico. Eliminarla por un `Km_anuals` vacío perdería el 69 % del dataset sin necesidad.
- **Salarios `n/d`:** la fila sigue siendo útil para otros análisis

In [276]:
grupos = {
    "1. Identificadores (no imputables)":   ["DNI", "Nom", "Cognoms"],
    "2. Obligatorios para el análisis":     ["País d'origen", "Ciutat", "Dia de Naixement", "Mes de Naixement",
                                             "Any de Naixement", "Salari mensual", "Grup Professional", "Càrrec"],
    "3. Opcionales / condicionales":        ["Gènere", "Fills", "No Fills", "Nombre_fills", "Te_cotxe",
                                             "Km_anuals", "Consum_mitja_L_100km", "Temperatura_mitjana_ciutat"],
}
tabla = pd.DataFrame([{"grupo": g, "campo": c, "nulos": int(df[c].isna().sum()), "% nulos": round(df[c].isna().mean() * 100, 2)} for g, cs in grupos.items() for c in cs])
display(tabla)

identificador = int(df[grupos["1. Identificadores (no imputables)"]].isna().any(axis=1).sum())
print(f"\nFilas sin algún identificador: {identificador} ")

obligatorio = int(df[grupos["2. Obligatorios para el análisis"]].isna().any(axis=1).sum())
print(f"\nFilas sin algún campo obligatorio: {obligatorio}")

sin_cifras = ~df["Salari mensual"].astype(str).str.contains(r"\d")
cant_cifras = int(sin_cifras.sum())
dicc_cifras = df.loc[sin_cifras, "Salari mensual"].value_counts().to_dict()
print(f"\nSalarios sin ninguna cifra (nulo disfrazado): en total {cant_cifras} : {dicc_cifras}")

,grupo,campo,nulos,% nulos
0,1. Identificadores (no imputables),DNI,0,0.00
1,1. Identificadores (no imputables),Nom,0,0.00
2,1. Identificadores (no imputables),Cognoms,0,0.00
3,2. Obligatorios para el análisis,País d'origen,0,0.00
4,2. Obligatorios para el análisis,Ciutat,0,0.00
5,2. Obligatorios para el análisis,Dia de Naixement,0,0.00
6,2. Obligatorios para el análisis,Mes de Naixement,0,0.00
7,2. Obligatorios para el análisis,Any de Naixement,0,0.00
8,2. Obligatorios para el análisis,Salari mensual,0,0.00
9,2. Obligatorios para el análisis,Grup Professional,0,0.00



Filas sin algún identificador: 0 

Filas sin algún campo obligatorio: 0

Salarios sin ninguna cifra (nulo disfrazado): en total 36 : {'n/d': 14, 'mil vuit-cents': 9, 'dos mil': 7, 'tres mil': 6}


## 6. Formatos incorrectos, valores imposibles e incoherencias (solo detectar)

**Qué observo:** el dataset contiene valores imposibles o incoherentes en campos concretos: fechas de nacimiento (día, mes y año inválidos a la vez), salarios con formatos mixtos y algunos escritos en texto, kilómetros negativos o de 1.000.000, consumos y temperaturas en valores "tope", y contradicciones entre `Fills`, `No Fills` y `Nombre_fills`.

**Decisión: estas filas se conservan.** Están bien escritas pero **no son útiles tal cual**. Aun así, el problema afecta a **un campo**, mientras que el resto del registro (identidad válida, trabajo, salario...) sí es utilizable. Eliminarlas haría perder información fiable


In [277]:
dia_mes_mal = ~df["Dia de Naixement"].between(1, 31) | ~df["Mes de Naixement"].between(1, 12)

anios = sorted(df["Any de Naixement"].dropna().unique())
anios_set = set(anios)
anios_sueltos = [anio for anio in anios if (anio - 1 not in anios_set) and (anio + 1 not in anios_set)]
anio_mal = df["Any de Naixement"].isin(anios_sueltos)

salario = df["Salari mensual"].astype(str).str.replace(r"\d", "9", regex=True).value_counts().to_string()
print("Perfil de formatos del salario (cada cifra → 9):")
print(salario)

validaciones = pd.Series({
    "Fecha de nacimiento imposible (día fuera de 1-31 o mes fuera de 1-12)": dia_mes_mal.sum(),
    f"Año de nacimiento suelto (sin año vecino): {sorted(anios_sueltos)}":    anio_mal.sum(),
    "Filas con fecha imposible Y año suelto a la vez":                   (dia_mes_mal & anio_mal).sum(),
    "Salario sin ninguna cifra ('n/d', texto)":                              sin_cifras.sum(),
    "Km anuales negativos":                                                  (df["Km_anuals"] < 0).sum(),
    "Km anuales = 1.000.000":                                                (df["Km_anuals"] >= 1e6).sum(),
    "Consumo exactamente 0,5 o 25,0 L/100 km":                               df["Consum_mitja_L_100km"].isin([0.5, 25.0]).sum(),
    "Temperatura en -10 / -5 / 38 / 42 °C":                                  df["Temperatura_mitjana_ciutat"].isin([-10, -5, 38, 42]).sum(),
    "Fills y No Fills marcados a la vez":                                    (df["Fills"].eq(True) & df["No Fills"].eq(True)).sum(),
    "Fills=True pero Nombre_fills = 0":                                      (df["Fills"].eq(True) & df["Nombre_fills"].eq(0)).sum(),
    "No Fills=True pero Nombre_fills > 0":                                   (df["No Fills"].eq(True) & (df["Nombre_fills"] > 0)).sum(),
}).astype(int)

display(validaciones)



Perfil de formatos del salario (cada cifra → 9):
Salari mensual
9,999 €           490
9999€             135
999 €             111
9999 euros        105
9.999 €            57
999€               34
999 euros          17
n/d                14
mil vuit-cents      9
dos mil             7
tres mil            6


,0
Fecha de nacimiento imposible (día fuera de 1-31 o mes fuera de 1-12),77
"Año de nacimiento suelto (sin año vecino): [np.float64(1940.0), np.float64(2019.0), np.float64(2026.0), np.float64(3000.0)]",77
Filas con fecha imposible Y año suelto a la vez,77
"Salario sin ninguna cifra ('n/d', texto)",36
Km anuales negativos,6
Km anuales = 1.000.000,1
"Consumo exactamente 0,5 o 25,0 L/100 km",119
Temperatura en -10 / -5 / 38 / 42 °C,65
Fills y No Fills marcados a la vez,46
Fills=True pero Nombre_fills = 0,32


## 7. Resumen de las decisiones y dataset resultante
- Codificación de nombres de columna y valores  `0`  Se repara el texto; no es un motivo de eliminación
- Fila completamente vacía `1` No contiene información
- Duplicados exactos `4` Mismo registro repetido; se conserva la 1ª aparición -  DNI inexistente (fila de prueba) `1` Valores de relleno: irrecuperable y no sustituible
- DNI duplicado con datos contradictorios `16` El identificador no es fiable y no hay criterio para elegir la versión correcta; imputarlo sería inventar datos

 **Total :**  **22** (2,2 %)

 **Dimensiones :**  antes **1.007 filas**, Despues **985 filas**


In [278]:
resumen = pd.DataFrame(registro, columns=["Paso / motivo", "Filas eliminadas"])
resumen["Filas restantes"] = len(df_raw) - resumen["Filas eliminadas"].cumsum()
display(resumen)

df_limpio = df.reset_index(drop=True)
assert df_limpio["DNI"].is_unique
assert df_limpio.duplicated(subset=cols_dato).sum() == 0
assert df_limpio[grupos["1. Identificadores (no imputables)"]].notna().all().all()

print(f"Filas iniciales: {len(df_raw)} → finales: {len(df_limpio)}  (eliminadas: {len(df_raw) - len(df_limpio)}, "
      f"{(len(df_raw) - len(df_limpio)) / len(df_raw):.1%})")

print("\nFilas descartadas (trazabilidad):")
display(pd.concat(descartados).sort_values("idx_origen")[["idx_origen", "Nom", "Cognoms", "DNI", "motivo"]])

,Paso / motivo,Filas eliminadas,Filas restantes
0,Fila completamente vacia,1,1006
1,Duplicado exacto (se conserva la 1ª aparición),4,1002
2,DNI inexistente: fila de prueba (valores de re...,1,1001
3,DNI duplicado con información contradictoria/n...,16,985


Filas iniciales: 1007 → finales: 985  (eliminadas: 22, 2.2%)

Filas descartadas (trazabilidad):


,idx_origen,Nom,Cognoms,DNI,motivo
25,25,Sara,Serra,90585882A,DNI duplicado con información contradictoria/n...
89,89,Guillem,Pujol Costa,54013571W,DNI duplicado con información contradictoria/n...
104,104,Oriol,Vila Casals,90585882A,DNI duplicado con información contradictoria/n...
114,114,Pol,Serra,90585882A,DNI duplicado con información contradictoria/n...
142,142,Marta,Ruiz,54013571W,DNI duplicado con información contradictoria/n...
250,250,Laia,Navarro Vila,15909601K,DNI duplicado con información contradictoria/n...
281,281,Xavier,Vidal,90585882A,DNI duplicado con información contradictoria/n...
558,558,Alexia,Pérez Díaz,49529438G,DNI duplicado con información contradictoria/n...
604,604,Laia,Navarro,15909601K,DNI duplicado con información contradictoria/n...
654,654,Oriol,Gómez,16618423V,DNI duplicado con información contradictoria/n...


# Ejercicio 3: Transformaciones necesarias y preparación del dataset
🎯 Objetivo: preparar el dataset para que sea coherente, analizable y estructuralmente correcto antes de realizar cualquier análisis descriptivo.
En este ejercicio deberás aplicar transformaciones básicas pero esenciales, orientadas a mejorar la calidad de los datos y facilitar el posterior análisis.


Aspectos a revisar y transformar:

- » ¿Los valores categóricos están normalizados y son consistentes?
- » ¿ Los valores nulos son coherentes con la información disponible?
- » ¿Existe redundancia de variables que pueda generar problemas analíticos?
- » ¿ Las variables contextuales (como temperatura, salario) están completas y preparadas para el análisis?

Concretamente, deberás:

- » Normalizar los valores categóricos para evitar ambigüedades (por ejemplo, valores que representan el mismo concepto pero están escritos de forma diferente).
- » Revisar la coherencia de los valores nulos, asegurando que:
Las personas con kilometraje anual y consumo medio asociados disponen de coche. En los casos en que conste consumo medio pero no kilometraje anual, será necesario decidir cómo resolver esta incoherencia de forma justificada.
Las personas que tienen un valor en el número de hijos tienen efectivamente hijos.
- » Unificar la información sobre hijos en una única columna, ya que mantener varias columnas con información complementaria no es óptimo y puede generar problemas analíticos (redundancia o multicolinealidad).
- » Validar la temperatura media. En caso de que esa información falte, deberás imputarla utilizando un criterio razonable (por ciudad o por un nivel superior justificado).
- » Revisar e imputar los salarios faltantes, aplicando criterios analíticos coherentes (como cargo, grupo profesional, ciudad, género o franja de edad).

## 1. Normalización de valores categóricos

### 1.1 Género

**Qué observo:** el mismo concepto tenía hasta 3 códigos distintos (`D`/`Dona`/`F` para mujer; `H`/`Home`/`M` para hombre), más `A`, `NC` y un `??` sin sentido.

**Decisión:** se agrupan en 4 categorías (`Dona`, `Home`, `Altre`, `No consta`) más los nulos ya existentes. El valor `??` se convierte en nulo porque no corresponde a ningún código documentado y no hay forma de saber qué significaba: inventar una categoría sería peor que dejarlo vacío.

In [279]:
MAPA_GENERE = {
    "D": "Dona", "Dona": "Dona", "F": "Dona",
    "H": "Home", "Home": "Home", "M": "Home",
    "A": "Altre",
    "NC": "No consta",
    "??": np.nan,
}
df["Gènere"] = df["Gènere"].map(MAPA_GENERE).fillna(df["Gènere"].where(df["Gènere"].isna()))

print("\nValores normalizados:")
print(df["Gènere"].value_counts(dropna=False))


Valores normalizados:
Gènere
Dona         541
Home         358
NaN           39
Altre         24
No consta     23
Name: count, dtype: int64


### 1.2 Cargo (Càrrec)

**Qué observo:** varios cargos estaban escritos de formas distintas para el mismo puesto: mayúsculas/minúsculas (`Data Analyst`/`Data analyst`), abreviaturas (`Admin.`/`Administratiu`), sinónimos (`Project lead`/`Cap de projecte`/`Cap de Projecte`) y variantes de idioma (`Analista`/`Analyst`).

**Decisión:** se unifican en 8 categorías, distinguiendo los niveles reales del puesto (`Analista` / `Analista Junior` / `Analista Senior` se mantienen separados porque son niveles distintos, no errores de escritura).

In [280]:
MAPA_CARREC = {
    "data analyst":        "Data Analyst",
    "analista":             "Analista",
    "analyst":               "Analista",
    "analista junior":      "Analista Junior",
    "senior analyst":       "Analista Senior",
    "tècnic it":            "Tècnic IT",
    "responsable de vendes":"Responsable de Vendes",
    "cap de projecte":      "Cap de Projecte",
    "project lead":         "Cap de Projecte",
    "administratiu":        "Administratiu",
    "admin.":               "Administratiu",
}
df["Càrrec"] = df["Càrrec"].str.strip().str.lower().map(MAPA_CARREC)

print("\nValores normalizados:")
print(df["Càrrec"].value_counts(dropna=False).to_string())


Valores normalizados:
Càrrec
Data Analyst             156
Tècnic IT                128
Cap de Projecte          123
Analista Senior          122
Responsable de Vendes    121
Analista                 121
Administratiu            108
Analista Junior          106


## 2. Coherencia: coche, kilómetros y consumo

**Qué observo:** 4 filas tenían kilómetros y/o consumo registrados pero `Te_cotxe = False`, una contradicción directa, Además, hay 2 filas con consumo medio pero sin kilometraje anual.

**Decisión:**
- Las 4 filas contradictorias se corrigen a `Te_cotxe = True`: si hay datos de uso del coche, la persona tiene coche.
- Para las 2 filas con consumo pero sin km: **ya tenían `Te_cotxe = True`**, así que no hay incoherencia en la columna del coche. El consumo medio (L/100 km) es una característica del vehículo, no depende de cuántos km se conduzcan al año, así que es razonable que exista sin el dato de kilometraje. **No se inventa** un valor de `Km_anuals`: se deja como nulo, porque no hay ninguna otra variable de la que deducirlo

In [281]:
tiene_km = df["Km_anuals"].notna()
tiene_consum = df["Consum_mitja_L_100km"].notna()
tiene_uso = tiene_km | tiene_consum

incoherentes = tiene_uso & (df["Te_cotxe"] != True)
print(f"\nFilas con km y/o consumo pero Te_cotxe no es True: {incoherentes.sum()}")
display(df.loc[incoherentes, ["Te_cotxe", "Km_anuals", "Consum_mitja_L_100km"]])

solo_consum = tiene_consum & ~tiene_km
print(f"\nCasos con consumo pero SIN kilometraje (la incoherencia que pide revisar el enunciado): {solo_consum.sum()}")
display(df.loc[solo_consum, ["Te_cotxe", "Km_anuals", "Consum_mitja_L_100km"]])


Filas con km y/o consumo pero Te_cotxe no es True: 4


,Te_cotxe,Km_anuals,Consum_mitja_L_100km
270,False,0.0,NaN
698,False,0.0,NaN
743,False,0.0,NaN
922,False,0.0,NaN



Casos con consumo pero SIN kilometraje (la incoherencia que pide revisar el enunciado): 2


,Te_cotxe,Km_anuals,Consum_mitja_L_100km
4,True,NaN,11.8
60,True,NaN,25.0


## 3. Unificación de la información sobre hijos en una única columna

**Qué observo:** `Fills`, `No Fills` y `Nombre_fills` decían tres cosas distintas del mismo hecho, y se contradecían entre sí (46 filas con ambas banderas a la vez, 48 sin ninguna).

**Decisión:** se conserva una única columna, `Nombre_fills`, porque es la más informativa (un número, no solo sí/no).
- Si ya tenía un valor, se respeta tal cual (aunque contradiga a `Fills`/`No Fills`): es el dato más preciso.
- Si faltaba pero `No Fills=True` (sin `Fills=True`), se completa con 0: es una deducción directa, no una imputación estadística.
- En el resto de casos (sin ninguna bandera, o con las dos a la vez y sin número) **se deja como nulo**: no hay ninguna base para decidir un número concreto, y rellenarlo con una media sería inventar un dato demográfico, no resolver una incoherencia.

In [282]:
F  = df["Fills"] == True
NF = df["No Fills"] == True
nf = df["Nombre_fills"]

print("Antes de unificar:")
print("Nombre_fills nulo:", nf.isna().sum())
print("Fills y No Fills marcados a la vez:", (F & NF).sum())
print("Ni Fills ni No Fills marcados:", (~F & ~NF).sum())

nf_final = nf.copy()
solo_no_fills = NF & ~F & nf.isna()
nf_final[solo_no_fills] = 0

df["Nombre_fills"] = nf_final
df = df.drop(columns=["Fills", "No Fills"])

print("\nDespués de unificar (columna única 'Nombre_fills'):")
print(df["Nombre_fills"].value_counts(dropna=False).sort_index())

Antes de unificar:
Nombre_fills nulo: 294
Fills y No Fills marcados a la vez: 46
Ni Fills ni No Fills marcados: 48

Después de unificar (columna única 'Nombre_fills'):
Nombre_fills
0.0    563
1.0    170
2.0     79
3.0     64
4.0     77
NaN     32
Name: count, dtype: int64


## 4. Validación e imputación de la temperatura media de la ciudad

**Qué observo:** la temperatura debería ser prácticamente constante dentro de cada ciudad (es una propiedad del clima local), pero antes de tocar nada la dispersión era enorme por culpa de los 4 valores centinela identificados en el Ejercicio 1 (`-10, -5, 38, 42`), que aparecían repartidos en 28 ciudades distintas sin relación con su clima real. Además había 66 huecos ya nulos de origen.

**Decisión:**
1. Los valores centinela se invalidan (pasan a nulo), porque son errores de captura, no clima real.
2. Todos los huecos (los originales + los invalidados) se imputan con la **media de esa misma ciudad**, calculada con sus valores fiables
3. Como red de seguridad se deja la media global para el caso de una ciudad sin ningún valor fiable.

In [283]:
print("Nulos actuales:", df["Temperatura_mitjana_ciutat"].isna().sum())
stats_ciudad = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].agg(["count", "nunique", "std"])

print("\nCiudades con más dispersión (antes de corregir):")
display(stats_ciudad.sort_values("std", ascending=False).head(5))

Nulos actuales: 66

Ciudades con más dispersión (antes de corregir):


,count,nunique,std
Ciutat,,,
Frankfurt,6,6,15.307830
Bergen,10,9,12.939672
Torino,15,12,12.519463
Düsseldorf,5,5,9.562426
Toulouse,14,14,9.562381


In [284]:
CENTINELA = [-10, -5, 38, 42]
son_erroneos = df["Temperatura_mitjana_ciutat"].isin(CENTINELA)
print("Valores centinela detectados:", son_erroneos.sum())

df.loc[son_erroneos, "Temperatura_mitjana_ciutat"] = np.nan
print("Nulos tras invalidar los centinela:", df["Temperatura_mitjana_ciutat"].isna().sum())

stats_ok = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].agg(["count", "mean", "std"])
print("\nDespués de invalidar los centinela (ejemplo, 5 ciudades):")
display(stats_ok.head(5))

print("\nDispersión media entre ciudades (antes ruidosa, ahora):", stats_ok["std"].mean().round(2), "°C — coherente con variación normal de mediciones")

print("\n¿Alguna ciudad se ha quedado sin ningún valor válido?", (stats_ok["count"] == 0).sum())

Valores centinela detectados: 65
Nulos tras invalidar los centinela: 131

Después de invalidar los centinela (ejemplo, 5 ciudades):


,count,mean,std
Ciutat,,,
A Coruña,40,14.240000,2.145454
Alacant,34,18.323529,1.869704
Barcelona,46,16.369565,2.047098
Bergen,7,8.028571,1.808972
Berlin,6,9.533333,1.500222



Dispersión media entre ciudades (antes ruidosa, ahora): 2.07 °C — coherente con variación normal de mediciones

¿Alguna ciudad se ha quedado sin ningún valor válido? 0


In [285]:
media_por_ciudad = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].transform("mean")
media_global = df["Temperatura_mitjana_ciutat"].mean()

antes = df["Temperatura_mitjana_ciutat"].isna().sum()
df["Temperatura_mitjana_ciutat"] = df["Temperatura_mitjana_ciutat"].fillna(media_por_ciudad)
df["Temperatura_mitjana_ciutat"] = df["Temperatura_mitjana_ciutat"].fillna(media_global)

print(f"Valores imputados: {antes} (huecos reales + centinela invalidados)")
print("\nNulos finales:", df['Temperatura_mitjana_ciutat'].isna().sum())
print("\nComprobación: ahora cada ciudad tiene una temperatura consistente:")
display(df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].agg(["mean", "std"]).round(2).head(6))

Valores imputados: 131 (huecos reales + centinela invalidados)

Nulos finales: 0

Comprobación: ahora cada ciudad tiene una temperatura consistente:


,mean,std
Ciutat,,
A Coruña,14.24,2.02
Alacant,18.32,1.66
Barcelona,16.37,1.89
Bergen,8.03,1.48
Berlin,9.53,1.50
Bilbao,13.90,2.37


## 5. Salario: normalización de formato e imputación de los que faltan

**Qué observo:** el salario mezclaba 5 formatos distintos y algunos valores en texto (`n/d`, `dos mil`...). Tras convertir todo a número, quedaban **14 filas sin ninguna cifra utilizable** (`n/d`).

**Decisión:** se imputan con la **mediana del salario en su mismo cargo** (`Càrrec`), el criterio más específico entre los disponibles (cargo, grupo, ciudad, género, edad) y con muestra suficiente en todos los casos (mínimo ~90 filas por cargo). Se usa la mediana y no la media para no dejarse arrastrar por salarios extremos.

In [286]:
NUMEROS_TEXTO = {"mil vuit-cents": 1800, "dos mil": 2000, "tres mil": 3000}

def a_numero(valor):
    """Convierte cualquiera de los formatos a un número (euros/mes)."""
    texto = str(valor).strip().lower()
    if texto in NUMEROS_TEXTO:
        return NUMEROS_TEXTO[texto]
    cifras = re.search(r"[\d\.,]+", texto)
    if not cifras:
        return np.nan
    limpio = cifras.group().replace(".", "").replace(",", "")
    return float(limpio)

df["Salari mensual"] = df["Salari mensual"].map(a_numero)

print("Salario convertido a número:")
print(df["Salari mensual"].describe().round(2))

print("\nValores que han quedado nulos (no tenían ninguna cifra):", df["Salari mensual"].isna().sum())

mediana_por_carrec = df.groupby("Càrrec")["Salari mensual"].transform("median")
df["Salari mensual"] = df["Salari mensual"].fillna(mediana_por_carrec)


print("\nMediana de salario por cargo :")
print(df.groupby("Càrrec")["Salari mensual"].median().round(2).sort_values(ascending=False))

print("\nNulos finales en salario:", df["Salari mensual"].isna().sum())

Salario convertido a número:
count     971.00
mean     1566.15
std       624.17
min       673.00
25%      1086.00
50%      1409.00
75%      1897.00
max      3672.00
Name: Salari mensual, dtype: float64

Valores que han quedado nulos (no tenían ninguna cifra): 14

Mediana de salario por cargo :
Càrrec
Cap de Projecte          1584.0
Analista Senior          1553.5
Data Analyst             1431.0
Tècnic IT                1397.0
Analista                 1325.5
Responsable de Vendes    1320.0
Administratiu            1297.0
Analista Junior          1219.0
Name: Salari mensual, dtype: float64

Nulos finales en salario: 0


In [287]:
print("Dimensiones finales:", df.shape)

print("\nNulos que quedan por columna:")
display(df.isna().sum()[lambda s: s > 0])

print("\nColumnas que se dejan deliberadamente sin imputar (no son 'huecos' a rellenar, sino ausencia de información real):")
print(f"  • Gènere ({df['Gènere'].isna().sum()}): la persona no indicó su género; no hay ninguna otra variable de la que deducirlo.")
print(f"  • Nombre_fills ({df['Nombre_fills'].isna().sum()}): sin ninguna base (ni bandera Fills/No Fills) para deducir el número de hijos.")
print(f"  • Te_cotxe ({df['Te_cotxe'].isna().sum()}) / Km_anuals ({df['Km_anuals'].isna().sum()}) / Consum ({df['Consum_mitja_L_100km'].isna().sum()}): "
      f"quien no tiene coche no tiene km ni consumo que reportar (nulo = 'no aplica', no un dato perdido); "
      f"imputar 'tiene coche' o un uso inventado no sería razonable.")

Dimensiones finales: (985, 18)

Nulos que quedan por columna:


,0
Gènere,39
Nombre_fills,32
Te_cotxe,342
Km_anuals,675
Consum_mitja_L_100km,728



Columnas que se dejan deliberadamente sin imputar (no son 'huecos' a rellenar, sino ausencia de información real):
  • Gènere (39): la persona no indicó su género; no hay ninguna otra variable de la que deducirlo.
  • Nombre_fills (32): sin ninguna base (ni bandera Fills/No Fills) para deducir el número de hijos.
  • Te_cotxe (342) / Km_anuals (675) / Consum (728): quien no tiene coche no tiene km ni consumo que reportar (nulo = 'no aplica', no un dato perdido); imputar 'tiene coche' o un uso inventado no sería razonable.


In [288]:
df_prep = df.reset_index(drop=True)
print("\nVista final:")
display(df_prep.head(5))


Vista final:


,idx_origen,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,1469.0,Grup B,Cap de Projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,Home,2718.0,Grup C,Analista Senior,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,Dona,1358.0,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,Home,1478.0,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,1284.0,Grup B,Administratiu,0.0,True,NaN,11.8,13.1


# Ejercicio 4: Primeros análisis descriptivos
🎯 Objetivo: obtener una primera visión global del dataset una vez aplicadas las transformaciones básicas, mediante estadísticas descriptivas y comparaciones simples entre grupos relevantes.
Aspectos a revisar y analizar:

- » Construir tablas resumen descriptivas.
- » Realizar comparaciones básicas entre grupos relevantes.

Concretamente, deberás:

  - » Construir tablas resumen descriptivas de salarios como una tabla resumen que permita ver el sueldo medio, medio, mínimo y máximo por Género. Ordena la tabla en función del sueldo medio.
  - » Crear una tabla resumen con el salario medio por género (filas) y país de origen (columnas). Añade las medias a los márgenes de la tabla. Aplica formato condicional a la tabla para ver en un color más intenso los valores más elevados.

## 1. Tabla resumen: salario por género

**Qué muestra:** el salario medio más alto lo tiene el grupo `Home`, seguido de `No consta`, `Altre` y `Dona`. Conviene tener en cuenta el tamaño de cada grupo (columna `n`): `Altre` y `No consta` son categorías mucho más pequeñas, así que su media es más sensible a valores puntuales.

In [289]:
resumen_genero = (
    df.groupby("Gènere")["Salari mensual"]
      .agg(salario_medio="mean", salario_mediano="median", salario_minimo="min", salario_maximo="max", n="count")
      .round(0)
      .sort_values("salario_medio", ascending=False)
)
display(resumen_genero)

resumen_genero.style.format("{:,.0f} €", na_rep="—").background_gradient(cmap="YlOrRd", axis=None)

,salario_medio,salario_mediano,salario_minimo,salario_maximo,n
Gènere,,,,,
Home,1694.0,1514.0,726.0,3672.0,358
Altre,1655.0,1206.0,673.0,3600.0,24
No consta,1571.0,1432.0,724.0,3600.0,23
Dona,1485.0,1378.0,701.0,3564.0,541


,salario_medio,salario_mediano,salario_minimo,salario_maximo,n
Gènere,,,,,
Home,"1,694 €","1,514 €",726 €,"3,672 €",358 €
Altre,"1,655 €","1,206 €",673 €,"3,600 €",24 €
No consta,"1,571 €","1,432 €",724 €,"3,600 €",23 €
Dona,"1,485 €","1,378 €",701 €,"3,564 €",541 €


## 2. Tabla resumen: salario medio por género y país de origen

**Qué muestra:** el mapa de calor resalta en tonos más intensos las combinaciones de salario medio más alto. El grupo `Altre` en Alemanya, Italia y Noruega aparece vacío (`—`) porque no hay ninguna persona de esa combinación en el dataset (ver la tabla de tamaños de grupo más abajo), no porque falte el dato.

In [290]:
piv_salario = df.pivot_table(
    index="Gènere", columns="País d'origen", values="Salari mensual",
    aggfunc="mean", margins=True, margins_name="Media"
).round(2)

display(piv_salario)

piv_salario.style.format("{:,.0f} €", na_rep="—").background_gradient(cmap="YlOrRd", axis=None)

País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,Media
Gènere,,,,,,
Altre,NaN,1734.07,1104.33,NaN,NaN,1655.35
Dona,1429.91,1483.44,1411.17,1469.40,1779.75,1485.48
Home,2103.82,1698.95,1668.69,1662.35,1479.00,1693.93
No consta,1634.00,1434.23,1461.83,2047.40,1237.00,1571.24
Media,1599.18,1574.02,1497.06,1567.73,1641.23,1570.76


País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,Media
Gènere,,,,,,
Altre,—,"1,734 €","1,104 €",—,—,"1,655 €"
Dona,"1,430 €","1,483 €","1,411 €","1,469 €","1,780 €","1,485 €"
Home,"2,104 €","1,699 €","1,669 €","1,662 €","1,479 €","1,694 €"
No consta,"1,634 €","1,434 €","1,462 €","2,047 €","1,237 €","1,571 €"
Media,"1,599 €","1,574 €","1,497 €","1,568 €","1,641 €","1,571 €"


#Nivel 2 - Análisis con datos enriquecidos y estructuras múltiples

#Ejercicio 1: Enriquecimiento mediante datasets externos
🎯 Objetivo: ampliar el dataset principal integrando información contextual externa y evaluar el impacto analítico de las decisiones de fusión.

Aspectos a revisar y analizar

- » ¿ Las diferentes fuentes de datos son coherentes y compatibles entre sí para poder calcular un salario ajustado?
- » ¿ La información contextual incorporada (coste de vida, políticas salariales) es suficiente para aplicar incrementos de forma consistente?
- » ¿Existen vacíos de información que impidan calcular correctamente el incremento salarial en algunos registros?
- » ¿Es necesario tomar decisiones analíticas cuando la información contextual es parcial o ausente?
- » ¿Cómo pueden estas decisiones influir en el cálculo del salario ajustado y su posterior interpretación?

Concretamente, tendrás que:

- » Integrar el dataset principal con:
El dataset contextual de ciudades.
La política salarial definida en formato JSON
- » Calcular un incremento salarial total combinando:
  - Grupo profesional.
  - Situación familiar.
  - Franja de edad.
  - Coste de vida de la ciudad.
- » Identificar qué registros no pueden recibir alguno de los ajustes y explicar su motivo.
- » Decidir, de modo razonado, cómo tratar estos casos en el cálculo del salario ajustado, justificando el impacto analítico de la decisión tomada.

### Carga de las fuentes

In [291]:

ciudades = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/phyton_especialidad/ciutats_context.csv", encoding="latin-1")

with open("/content/drive/MyDrive/Colab Notebooks/phyton_especialidad/politica_salarial.json", encoding="utf-8") as f:
    politica = json.load(f)


## 1. ¿Son coherentes y compatibles las fuentes entre sí?





### 1.1 Problemas de codificación en el CSV

el CSV está en latin-1, y además la fila de `A Coruña` no tiene el carácter `ñ` real sino el texto literal `\u00f1`

Se repara antes de cruzar nada, si no esta fila no encajaría con `A Coruña` del dataset principal.

In [292]:

fila_rara = ciudades[ciudades["Ciutat"].str.contains(r"\\u", regex=True)]
display(fila_rara)

def repara_escape(texto):
    """Arregla escapes unicode literales tipo \\u00f1 -> ñ, si los hay."""
    if isinstance(texto, str) and "\\u" in texto:
        return texto.encode("latin-1").decode("unicode_escape")
    return texto

ciudades["Ciutat"] = ciudades["Ciutat"].map(repara_escape)
print("Después de reparar:", ciudades.loc[fila_rara.index, "Ciutat"].tolist())

,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus
10,A Coru\u00f1a,Espanya,69,7.1,Oceànic


Después de reparar: ['A Coruña']


### 1.2 ¿Los países coinciden? — nombres de ciudad ambiguos

**Qué observo:** el CSV incluye 2 países (`Estats Units`, `Veneçuela`) que **no existen** en el dataset principal (que solo tiene Espanya, França, Itàlia, Alemanya y Noruega), y justo usa esos países para repetir nombres de ciudad que ya existen en los países buenos: `Paris` (França **y** Estats Units) y `Valencia`/`València` (Veneçuela y Espanya).

**Decisión:** el cruce (`merge`) se hace por **ciudad + país a la vez**. Si se cruzara solo por `Ciutat`, cada empleado de París (Francia) se emparejaría también con la fila de Estados Unidos, duplicando filas con un coste de vida que no le corresponde.

In [293]:
print("?Países en el dataset principal:", sorted(df["País d'origen"].unique()))

print("\nPaíses en el CSV de ciudades   :", sorted(ciudades["País"].unique()))
print()
repetidas = ciudades[ciudades.duplicated(subset="Ciutat", keep=False)].sort_values("Ciutat")
display(repetidas)

?Países en el dataset principal: ['Alemanya', 'Espanya', 'França', 'Itàlia', 'Noruega']

Países en el CSV de ciudades   : ['Alemanya', 'Espanya', 'Estats Units', 'França', 'Itàlia', 'Noruega', 'Veneçuela']



,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus
14,Paris,França,85,9.0,Oceànic
34,Paris,Estats Units,88,6.0,Continental


### 1.3 ¿Todas las ciudades del dataset principal tienen fila en el CSV?



**Qué observo:** tras reparar la codificación y normalizar acentos (con lo que `München`/`Munchen` ya emparejan bien), quedan **10 ciudades reales del dataset principal sin ninguna fila de contexto en el CSV**
-  el CSV solo cubre ciudades "grandes" de cada país y se olvida de las más pequeñas. Esto afecta a **130 empleados (13,2 %)**, sobre todo en Noruega (donde el CSV solo cubre 2 de sus 5 ciudades) y en Màlaga (53 personas, la ciudad española más numerosa que falta).

Se logra determinar que las fuentes **no son totalmente coherentes ni compatibles**; hay un vacío real de información contextual

In [294]:
def normaliza(texto):
    """Quita acentos y pasa a minúsculas."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower().strip()

df["_ciutat_norm"] = df["Ciutat"].map(normaliza)
ciudades["_ciutat_norm"] = ciudades["Ciutat"].map(normaliza)

paises_dataset = df["País d'origen"].unique().tolist()
ciudades_validas = ciudades[ciudades["País"].isin(paises_dataset)].copy()

pares_dataset = df[["_ciutat_norm", "País d'origen", "Ciutat"]].drop_duplicates()
pares_csv = ciudades_validas[["_ciutat_norm", "País"]].drop_duplicates().rename(columns={"País": "País d'origen"})

comparacion = pares_dataset.merge(pares_csv, on=["_ciutat_norm", "País d'origen"], how="left", indicator=True)
sin_contexto = comparacion[comparacion["_merge"] == "left_only"][["Ciutat", "País d'origen"]]

print(f"Ciudades del dataset principal sin fila en el CSV: {len(sin_contexto)} de {df['Ciutat'].nunique()}")
display(sin_contexto.sort_values(["País d'origen", "Ciutat"]))

empleados_sin_contexto = df.merge(sin_contexto, on=["Ciutat", "País d'origen"])
print(f"\nEmpleados afectados: {len(empleados_sin_contexto)} de {len(df)} ({len(empleados_sin_contexto) / len(df):.1%})")
display(empleados_sin_contexto.groupby(["País d'origen", "Ciutat"]).size().rename("empleados"))

Ciudades del dataset principal sin fila en el CSV: 10 de 44


,Ciutat,País d'origen
35,Düsseldorf,Alemanya
43,Köln,Alemanya
14,Màlaga,Espanya
38,Nantes,França
29,Nice,França
18,Toulouse,França
33,Bologna,Itàlia
7,Stavanger,Noruega
40,Tromsø,Noruega
26,Trondheim,Noruega



Empleados afectados: 130 de 985 (13.2%)


País d'origen  Ciutat    
Alemanya       Düsseldorf     5
               Köln           4
Espanya        Màlaga        53
França         Nantes        12
               Nice          11
               Toulouse      15
Itàlia         Bologna       11
Noruega        Stavanger      9
               Tromsø         4
               Trondheim      6
Name: empleados, dtype: int64

## 2. Integración del dataset principal con el CSV de ciudades

**Qué observo:** el `merge` no duplica ni pierde filas (sigue habiendo 985), y las 130 filas sin ciudad equivalente en el CSV quedan con `NaN` en `Cost_vida_index`, `Transport_public_score` y `Clima_tipus`

In [295]:
df = df.merge(
    ciudades_validas[["_ciutat_norm", "País", "Cost_vida_index", "Transport_public_score", "Clima_tipus"]]
        .rename(columns={"País": "País d'origen"}),
    on=["_ciutat_norm", "País d'origen"], how="left"
)
df = df.drop(columns="_ciutat_norm")

print("Nulos tras la integración (ciudades sin contexto):")
display(df[["Cost_vida_index", "Transport_public_score", "Clima_tipus"]].isna().sum())

assert len(df) == 985, "el merge no debe duplicar ni perder filas"
print("\nFilas del dataset tras integrar:", len(df), "(igual que antes → el merge 1 a 1 es correcto)")

Nulos tras la integración (ciudades sin contexto):


,0
Cost_vida_index,130
Transport_public_score,130
Clima_tipus,130



Filas del dataset tras integrar: 985 (igual que antes → el merge 1 a 1 es correcto)


## 3. Aplicación de la política salarial (JSON)

La política define 4 ajustes que se **suman** para formar el incremento total, más 2 reglas (`max_total_increment_percent`, `min_salary_threshold`). Tres de los ajustes (coste de vida, transporte) se definen por nivel (`low`/`medium`/`high`), pero el CSV da un número, no un nivel: hay que **convertir el número en nivel** antes de poder aplicar la política.

### 3.1 Convertir los índices numéricos en niveles bajo / medio / alto

Se usan los terciles (33 % / 66 %) de los valores disponibles en el CSV como puntos de corte

In [296]:

p33_cost, p66_cost = ciudades_validas["Cost_vida_index"].quantile([0.33, 0.66])
p33_tr,   p66_tr   = ciudades_validas["Transport_public_score"].quantile([0.33, 0.66])
print(f"Coste de vida  → bajo: <{p33_cost:.1f} | medio: {p33_cost:.1f}-{p66_cost:.1f} | alto: >{p66_cost:.1f}")
print(f"Transporte     → bajo: <{p33_tr:.2f} | medio: {p33_tr:.2f}-{p66_tr:.2f} | alto: >{p66_tr:.2f}")

def nivel(valor, p33, p66):
    if pd.isna(valor):
        return np.nan
    if valor <= p33:
        return "low"
    if valor <= p66:
        return "medium"
    return "high"

df["nivell_cost_vida"] = df["Cost_vida_index"].apply(nivel, args=(p33_cost, p66_cost))
df["nivell_transport"] = df["Transport_public_score"].apply(nivel, args=(p33_tr, p66_tr))

print("\nDistribución de niveles asignados:")
print(df["nivell_cost_vida"].value_counts(dropna=False))
print()
print(df["nivell_transport"].value_counts(dropna=False))

Coste de vida  → bajo: <70.0 | medio: 70.0-76.8 | alto: >76.8
Transporte     → bajo: <7.09 | medio: 7.09-7.60 | alto: >7.60

Distribución de niveles asignados:
nivell_cost_vida
low       473
medium    217
high      165
NaN       130
Name: count, dtype: int64

nivell_transport
low       401
medium    248
high      206
NaN       130
Name: count, dtype: int64


### 3.2 Franja de edad

**Qué observo:** 77 filas tienen una fecha de nacimiento imposible para esas filas no se puede calcular una edad real, así que su franja de edad queda como `NaN`.

In [297]:
anios = sorted(df["Any de Naixement"].dropna().unique())
anios_set = set(anios)
anios_sueltos = [anio for anio in anios if (anio - 1 not in anios_set) and (anio + 1 not in anios_set)]

fecha_valida = df["Dia de Naixement"].between(1, 31) & df["Mes de Naixement"].between(1, 12) & ~df["Any de Naixement"].isin(anios_sueltos)

print("Fechas de nacimiento válidas:", fecha_valida.sum(), "| inválidas:", (~fecha_valida).sum())

ANIO_REFERENCIA = 2025
edad = ANIO_REFERENCIA - df["Any de Naixement"]

df["edat"] = edad.where(fecha_valida)

def franja_edad(edad):
    if edad < 30: return "18-29"
    if edad < 45: return "30-44"
    if edad < 60: return "45-59"
    return "60+"

df["franja_edat"] = edad.where(fecha_valida).apply(lambda edad: franja_edad(edad) if pd.notna(edad) else np.nan)
print("\nDistribución de franjas de edad:")
print(df["franja_edat"].value_counts(dropna=False))

Fechas de nacimiento válidas: 908 | inválidas: 77

Distribución de franjas de edad:
franja_edat
30-44    293
45-59    279
18-29    176
60+      160
NaN       77
Name: count, dtype: int64


### 3.3 Situación familiar
- Se usa el tipo "boolean" anulable de pandas (no np.where): mezclar NaN con True/False
- en np.where los convierte de forma silenciosa a 1.0/0.0 (float)

In [298]:
df["te_fills"] = (df["Nombre_fills"] > 0).astype("boolean")
df.loc[df["Nombre_fills"].isna(), "te_fills"] = pd.NA

print(df["te_fills"].value_counts(dropna=False))
print("\nSin dato de hijos:", df['te_fills'].isna().sum())

te_fills
False    563
True     390
<NA>      32
Name: count, dtype: Int64

Sin dato de hijos: 32


### 3.4 Calcular el incremento total
**Qué observo:** el grupo profesional `inc_grup` está completo en las 985 filas

Los otros 4 componentes tienen huecos, y **cada uno por un motivo distinto**: 77 filas sin edad válida, 32 sin dato de hijos, y 130 sin contexto de ciudad (que afecta a la vez a coste de vida y transporte, porque vienen del mismo `merge`).

In [299]:
def busca(diccionario, clave):
    """Devuelve el valor de la política para esa clave, o NaN si la clave falta o es nula."""
    if pd.isna(clave):
        return np.nan
    return diccionario.get(clave, np.nan)

df["inc_grup"]      = df["Grup Professional"].map(politica["base_increment_by_group"])
df["inc_fills"]     = df["te_fills"].map({True: politica["children_adjustment"]["has_children"],
                                           False: politica["children_adjustment"]["no_children"]})
df["inc_edat"]      = df["franja_edat"].map(politica["age_adjustment"])
df["inc_cost_vida"] = df["nivell_cost_vida"].map(politica["cost_of_living_adjustment"])
df["inc_transport"] = df["nivell_transport"].map(politica["transport_adjustment"])

componentes = ["inc_grup", "inc_fills", "inc_edat", "inc_cost_vida", "inc_transport"]
print("Filas con CADA componente disponible (antes de decidir qué hacer con los que faltan):")
display(df[componentes].notna().sum())

Filas con CADA componente disponible (antes de decidir qué hacer con los que faltan):


,0
inc_grup,985
inc_fills,953
inc_edat,908
inc_cost_vida,855
inc_transport,855


## 4. Qué hacer con los ajustes que faltan

**Decisión y justificación:**

Para los ajustes que faltan, se asigna **0** , en vez de intentar adivinar un valor:
- No es una media ni una imputación estadística: es literalmente "no se aplica ni bonificación ni penalización" a esa persona por ese concepto, que es justo lo que corresponde cuando no se puede verificar el dato de origen.
- Es la opción **más conservadora**: todos los valores de la política son ≥ 0 (o 0, como `no_children`), así que asignar 0 **nunca sobreestima** el incremento de nadie. Elegir otra cosa (la media del grupo, el valor más frecuente...) inflaría artificialmente el salario ajustado de gente de la que no tenemos ese dato.
- **Cada fila afectada se marca explícitamente** en una columna `ajustos_incompletos`, para que cualquier análisis posterior (y cualquier persona que revise el resultado) sepa que ese incremento es parcial y no está calculado con toda la información.


**Impacto analítico de la decisión:** estas 226 personas (22,9 %) (algunas con más de un hueco a la vez) van a salir con un incremento igual o **inferior** al que les correspondería si se conociera toda su información.

In [300]:
resumen_gaps = pd.DataFrame({
    "componente": ["Grupo profesional", "Hijos", "Franja de edad", "Coste de vida", "Transporte"],
    "columna": componentes,
    "filas sin ese ajuste": [df[c].isna().sum() for c in componentes],
    "motivo": [
        "— (sin huecos)",
        "Fills/No Fills/Nombre_fills inconsistentes entre sí, sin ningún criterio para deducirlo (Ejercicio 3)",
        "Fecha de nacimiento imposible (día, mes o año inválido)",
        "Ciudad sin fila equivalente en el CSV de contexto",
        "Ciudad sin fila equivalente en el CSV de contexto",
    ],
})
display(resumen_gaps)

,componente,columna,filas sin ese ajuste,motivo
0,Grupo profesional,inc_grup,0,— (sin huecos)
1,Hijos,inc_fills,32,Fills/No Fills/Nombre_fills inconsistentes ent...
2,Franja de edad,inc_edat,77,"Fecha de nacimiento imposible (día, mes o año ..."
3,Coste de vida,inc_cost_vida,130,Ciudad sin fila equivalente en el CSV de contexto
4,Transporte,inc_transport,130,Ciudad sin fila equivalente en el CSV de contexto


In [301]:
df["ajustos_incompletos"] = df[componentes[1:]].isna().any(axis=1)
print("Personas con algún ajuste incompleto:", df['ajustos_incompletos'].sum(), f"({df['ajustos_incompletos'].mean():.1%})")

for c in componentes:
    df[c] = df[c].fillna(0)

Personas con algún ajuste incompleto: 226 (22.9%)


### 4.1 Incremento total, con el tope de la política

**Qué observo:** algunas combinaciones (p. ej. Grupo D + hijos + 45-59 años + ciudad cara) superan el 15 % antes de aplicar el tope.

**Decisión:** se aplica el tope tal como indica la política (`max_total_increment_percent`), recortando el incremento a 15 % como máximo, y se guarda en `tope_aplicat` quién se ha visto afectado, para que quede trazado.

In [302]:
df["increment_total_pct"] = df[componentes].sum(axis=1)

tope = politica["rules"]["max_total_increment_percent"]
superan_tope = df["increment_total_pct"] > tope
print(f"Personas que superan el tope del {tope}% antes de aplicarlo:", superan_tope.sum())

display(df.loc[superan_tope, componentes + ["increment_total_pct"]].head())

df["increment_aplicat_pct"] = df["increment_total_pct"].clip(upper=tope)
df["tope_aplicat"] = superan_tope

Personas que superan el tope del 15% antes de aplicarlo: 0


,inc_grup,inc_fills,inc_edat,inc_cost_vida,inc_transport,increment_total_pct


### 4.2 Umbral salarial mínimo (`min_salary_threshold`)

**Decisión:** la política no especifica qué incremento corresponde a los salarios por debajo del umbral, así que se calcula el incremento de estas 30 personas con la misma fórmula que al resto, y se marcan en una columna (`bajo_umbral_minim`)

In [303]:
umbral = politica["rules"]["min_salary_threshold"]
bajo_umbral = df["Salari mensual"] < umbral
print(f"Personas con salario actual por debajo del umbral ({umbral} €): {bajo_umbral.sum()}")
display(df.loc[bajo_umbral, "Salari mensual"].describe())

Personas con salario actual por debajo del umbral (800 €): 30


,Salari mensual
count,30.000000
mean,758.233333
std,32.141587
min,673.000000
25%,730.250000
50%,768.000000
75%,782.500000
max,797.000000


In [304]:
df["bajo_umbral_minim"] = bajo_umbral

df["salari_ajustat"] = (df["Salari mensual"] * (1 + df["increment_aplicat_pct"] / 100)).round(2)

display(df[["Nom", "Cognoms", "Càrrec", "Salari mensual", "increment_aplicat_pct",
            "salari_ajustat", "ajustos_incompletos", "tope_aplicat", "bajo_umbral_minim"]].head(5))

,Nom,Cognoms,Càrrec,Salari mensual,increment_aplicat_pct,salari_ajustat,ajustos_incompletos,tope_aplicat,bajo_umbral_minim
0,Joana,Gil Navarro,Cap de Projecte,1469.0,7.0,1571.83,False,False,False
1,Marc,Muñoz,Analista Senior,2718.0,4.0,2826.72,False,False,False
2,Noa,Serra,Tècnic IT,1358.0,8.5,1473.43,False,False,False
3,Pol,Gil,Data Analyst,1478.0,6.5,1574.07,False,False,False
4,David,Vila,Administratiu,1284.0,5.5,1354.62,False,False,False


In [305]:
print(f"Incremento medio aplicado: {df['increment_aplicat_pct'].mean():.2f} %")
print(f"Personas con el tope del {tope}% aplicado: {df['tope_aplicat'].sum()}")
print(f"Personas con algún ajuste incompleto (0 por falta de dato): {df['ajustos_incompletos'].sum()} ({df['ajustos_incompletos'].mean():.1%})")
print(f"Personas bajo el umbral salarial mínimo, fuera de lo que documenta la política: {df['bajo_umbral_minim'].sum()}")

df_enriquecido = df.reset_index(drop=True)
print("\nDataset enriquecido:", df_enriquecido.shape)

Incremento medio aplicado: 6.52 %
Personas con el tope del 15% aplicado: 0
Personas con algún ajuste incompleto (0 por falta de dato): 226 (22.9%)
Personas bajo el umbral salarial mínimo, fuera de lo que documenta la política: 30

Dataset enriquecido: (985, 37)


#Ejercicio 2: Reestructuración del dataset

Objetivo: preparar el dataset para análisis comparativos y/o visuales. Convertir información dispersa en formato largo:
Salarios
Movilidad
Contexto climático etc
Aspectos a revisar y analizar
- » ¿ Existen variables que representan conceptos similares o relacionados, pero están distribuidas en diferentes columnas?
- » El formato actual del dataset facilita o dificulta:
comparaciones entre magnitudes.
agregaciones por grupos.
- » ¿Qué impacto tiene la forma del dataset (ancho vs largo) en la lectura e interpretación de los datos?


Concretamente, tendrás que:

- » Reestructurar el dataset utilizando melt() para convertirlo a formato largo, definiendo previamente:
  - las columnas que actúan como identificadoras (por ejemplo: persona, ciudad, grupo profesional, género, etc.).
  - las columnas que pasan a representar valores medidos (por ejemplo: salarios, movilidad, variables contextuales).
- » Analizar por qué el formato largo es más adecuado para:
  - realizar comparaciones entre distintos tipos de variables.
  - calcular estadísticos agregados.

## 1. ¿Qué columnas son iguales pero están repartidas en varias columnas?

**Qué observo:** hay tres grupos de columnas que, en el fondo, representan **el mismo tipo de concepto** repetido con nombres distintos, lo que es la señal clásica de que el dataset está en formato ancho (*wide*) y se beneficiaría de pasar a formato largo (*long*):

 - **Salarios:** `Salari mensual` (antes del incremento) y `salari_ajustat` (después) son dos versiones de la misma magnitud (euros/mes).
 - **Movilidad:** `Km_anuals` y `Consum_mitja_L_100km` son dos variables distintas del mismo ámbito (uso del coche).
 - **Contexto de la ciudad:** `Cost_vida_index`, `Transport_public_score` y `Temperatura_mitjana_ciutat` son tres variables numéricas que describen a la ciudad, no a la persona.

 Además, los 5 componentes del incremento (`inc_grup`, `inc_fills`, `inc_edat`, `inc_cost_vida`, `inc_transport`) son también la misma magnitud (puntos porcentuales) repetida en 5 columnas.

In [306]:
print("\nTodas las columnas del dataset enriquecido:")
for i, c in enumerate(df_enriquecido.columns):
    print(f"  {i:>2}. {c}")


Todas las columnas del dataset enriquecido:
   0. idx_origen
   1. Nom
   2. Cognoms
   3. DNI
   4. País d'origen
   5. Ciutat
   6. Dia de Naixement
   7. Mes de Naixement
   8. Any de Naixement
   9. Gènere
  10. Salari mensual
  11. Grup Professional
  12. Càrrec
  13. Nombre_fills
  14. Te_cotxe
  15. Km_anuals
  16. Consum_mitja_L_100km
  17. Temperatura_mitjana_ciutat
  18. Cost_vida_index
  19. Transport_public_score
  20. Clima_tipus
  21. nivell_cost_vida
  22. nivell_transport
  23. edat
  24. franja_edat
  25. te_fills
  26. inc_grup
  27. inc_fills
  28. inc_edat
  29. inc_cost_vida
  30. inc_transport
  31. ajustos_incompletos
  32. increment_total_pct
  33. increment_aplicat_pct
  34. tope_aplicat
  35. bajo_umbral_minim
  36. salari_ajustat


## 2. Columnas identificadoras (`id_vars`) y columnas de valor (`value_vars`)

**Criterio usado:** una columna es **identificadora** si describe *quién es* la persona o *dónde/en qué grupo* está (no cambia según la magnitud que se mida); es **valor** si es una magnitud medida que se puede comparar con otras magnitudes parecidas. No se incluyen en `id_vars` columnas que ya son "banderas" de calidad (`ajustos_incompletos`, `tope_aplicat`...), porque no aportan a la comparación entre magnitudes y solo añadirían ruido a la tabla larga.

In [307]:
id_vars = [
    "DNI", "Nom", "Cognoms", "Gènere", "País d'origen", "Ciutat",
    "Grup Professional", "Càrrec", "franja_edat", "te_fills", "Clima_tipus",
]

grupos_valor = {
    "salarios":   ["Salari mensual", "salari_ajustat"],
    "movilidad":  ["Km_anuals", "Consum_mitja_L_100km"],
    "contexto_ciudad": ["Cost_vida_index", "Transport_public_score", "Temperatura_mitjana_ciutat"],
    "componentes_incremento": ["inc_grup", "inc_fills", "inc_edat", "inc_cost_vida", "inc_transport"],
}

print("Identificadoras:", id_vars)
print("\nGrupos de columnas que pasarán a formato largo:")
for grupo, cols in grupos_valor.items():
    print(f"  {grupo}: {cols}")

Identificadoras: ['DNI', 'Nom', 'Cognoms', 'Gènere', "País d'origen", 'Ciutat', 'Grup Professional', 'Càrrec', 'franja_edat', 'te_fills', 'Clima_tipus']

Grupos de columnas que pasarán a formato largo:
  salarios: ['Salari mensual', 'salari_ajustat']
  movilidad: ['Km_anuals', 'Consum_mitja_L_100km']
  contexto_ciudad: ['Cost_vida_index', 'Transport_public_score', 'Temperatura_mitjana_ciutat']
  componentes_incremento: ['inc_grup', 'inc_fills', 'inc_edat', 'inc_cost_vida', 'inc_transport']


## 3. Formato largo: salarios (antes / después del incremento)

In [308]:
largo_salarios = df_enriquecido.melt(
    id_vars=id_vars,
    value_vars=grupos_valor["salarios"],
    var_name="tipo_salario",
    value_name="euros_mes",
)
largo_salarios["tipo_salario"] = largo_salarios["tipo_salario"].map({
    "Salari mensual": "Salario base", "salari_ajustat": "Salario ajustado",
})

print("Formato ancho:", df_enriquecido[id_vars + grupos_valor["salarios"]].shape)
print("Formato largo:", largo_salarios.shape)
display(largo_salarios.head(6))

Formato ancho: (985, 13)
Formato largo: (1970, 13)


,DNI,Nom,Cognoms,Gènere,País d'origen,Ciutat,Grup Professional,Càrrec,franja_edat,te_fills,Clima_tipus,tipo_salario,euros_mes
0,66722344X,Joana,Gil Navarro,Dona,Espanya,Valladolid,Grup B,Cap de Projecte,60+,True,Continental,Salario base,1469.0
1,48840994W,Marc,Muñoz,Home,Espanya,Alacant,Grup C,Analista Senior,60+,False,Mediterrani,Salario base,2718.0
2,14308421X,Noa,Serra,Dona,Espanya,Alacant,Grup A,Tècnic IT,60+,True,Mediterrani,Salario base,1358.0
3,58586340F,Pol,Gil,Home,Espanya,Sevilla,Grup B,Data Analyst,30-44,True,Mediterrani,Salario base,1478.0
4,82070937P,David,Vila,Dona,Espanya,Bilbao,Grup B,Administratiu,60+,False,Oceànic,Salario base,1284.0
5,25014631B,Alexia,Puig,Dona,Espanya,Las Palmas de Gran Canaria,Grup B,Cap de Projecte,45-59,False,Subtropical,Salario base,2069.0


## 4. Formato largo: movilidad

In [309]:
largo_movilidad = df_enriquecido.melt(
    id_vars=id_vars,
    value_vars=grupos_valor["movilidad"],
    var_name="variable_movilidad",
    value_name="valor",
)
largo_movilidad["variable_movilidad"] = largo_movilidad["variable_movilidad"].map({
    "Km_anuals": "Km anuales", "Consum_mitja_L_100km": "Consumo (L/100km)",
})
display(largo_movilidad.head(6))

,DNI,Nom,Cognoms,Gènere,País d'origen,Ciutat,Grup Professional,Càrrec,franja_edat,te_fills,Clima_tipus,variable_movilidad,valor
0,66722344X,Joana,Gil Navarro,Dona,Espanya,Valladolid,Grup B,Cap de Projecte,60+,True,Continental,Km anuales,32108.0
1,48840994W,Marc,Muñoz,Home,Espanya,Alacant,Grup C,Analista Senior,60+,False,Mediterrani,Km anuales,19496.0
2,14308421X,Noa,Serra,Dona,Espanya,Alacant,Grup A,Tècnic IT,60+,True,Mediterrani,Km anuales,NaN
3,58586340F,Pol,Gil,Home,Espanya,Sevilla,Grup B,Data Analyst,30-44,True,Mediterrani,Km anuales,NaN
4,82070937P,David,Vila,Dona,Espanya,Bilbao,Grup B,Administratiu,60+,False,Oceànic,Km anuales,NaN
5,25014631B,Alexia,Puig,Dona,Espanya,Las Palmas de Gran Canaria,Grup B,Cap de Projecte,45-59,False,Subtropical,Km anuales,NaN


## 5. Formato largo: contexto climático y de ciudad

In [310]:
largo_contexto = df_enriquecido.melt(
    id_vars=id_vars,
    value_vars=grupos_valor["contexto_ciudad"],
    var_name="variable_ciudad",
    value_name="valor",
)
largo_contexto["variable_ciudad"] = largo_contexto["variable_ciudad"].map({
    "Cost_vida_index": "Índice coste de vida", "Transport_public_score": "Transporte público",
    "Temperatura_mitjana_ciutat": "Temperatura media",
})
display(largo_contexto.head(6))

,DNI,Nom,Cognoms,Gènere,País d'origen,Ciutat,Grup Professional,Càrrec,franja_edat,te_fills,Clima_tipus,variable_ciudad,valor
0,66722344X,Joana,Gil Navarro,Dona,Espanya,Valladolid,Grup B,Cap de Projecte,60+,True,Continental,Índice coste de vida,64.0
1,48840994W,Marc,Muñoz,Home,Espanya,Alacant,Grup C,Analista Senior,60+,False,Mediterrani,Índice coste de vida,68.0
2,14308421X,Noa,Serra,Dona,Espanya,Alacant,Grup A,Tècnic IT,60+,True,Mediterrani,Índice coste de vida,68.0
3,58586340F,Pol,Gil,Home,Espanya,Sevilla,Grup B,Data Analyst,30-44,True,Mediterrani,Índice coste de vida,65.0
4,82070937P,David,Vila,Dona,Espanya,Bilbao,Grup B,Administratiu,60+,False,Oceànic,Índice coste de vida,72.0
5,25014631B,Alexia,Puig,Dona,Espanya,Las Palmas de Gran Canaria,Grup B,Cap de Projecte,45-59,False,Subtropical,Índice coste de vida,62.0


## 6. Formato largo: componentes del incremento salarial

In [311]:
largo_incremento = df_enriquecido.melt(
    id_vars=id_vars,
    value_vars=grupos_valor["componentes_incremento"],
    var_name="componente",
    value_name="puntos_pct",
)
largo_incremento["componente"] = largo_incremento["componente"].map({
    "inc_grup": "Grupo profesional", "inc_fills": "Hijos", "inc_edat": "Edad",
    "inc_cost_vida": "Coste de vida", "inc_transport": "Transporte",
})
display(largo_incremento.head(6))

print("\nComprobación: sumando las 5 filas de cada persona se recupera el incremento total")
comprobacion = largo_incremento.groupby("DNI")["puntos_pct"].sum().round(2).sort_index()
original = df_enriquecido.set_index("DNI")["increment_total_pct"].round(2).sort_index()
print("¿Coinciden todas las personas?", comprobacion.equals(original))

,DNI,Nom,Cognoms,Gènere,País d'origen,Ciutat,Grup Professional,Càrrec,franja_edat,te_fills,Clima_tipus,componente,puntos_pct
0,66722344X,Joana,Gil Navarro,Dona,Espanya,Valladolid,Grup B,Cap de Projecte,60+,True,Continental,Grupo profesional,3.5
1,48840994W,Marc,Muñoz,Home,Espanya,Alacant,Grup C,Analista Senior,60+,False,Mediterrani,Grupo profesional,2.0
2,14308421X,Noa,Serra,Dona,Espanya,Alacant,Grup A,Tècnic IT,60+,True,Mediterrani,Grupo profesional,5.0
3,58586340F,Pol,Gil,Home,Espanya,Sevilla,Grup B,Data Analyst,30-44,True,Mediterrani,Grupo profesional,3.5
4,82070937P,David,Vila,Dona,Espanya,Bilbao,Grup B,Administratiu,60+,False,Oceànic,Grupo profesional,3.5
5,25014631B,Alexia,Puig,Dona,Espanya,Las Palmas de Gran Canaria,Grup B,Cap de Projecte,45-59,False,Subtropical,Grupo profesional,3.5



Comprobación: sumando las 5 filas de cada persona se recupera el incremento total
¿Coinciden todas las personas? True


## 7. ¿Por qué el formato largo facilita comparar y agregar?

**Por qué es más adecuado el formato largo:**
- **Para comparar magnitudes del mismo tipo** (salario base vs ajustado; los 5 componentes del incremento), el formato largo las deja en **una única columna** (`euros_mes`, `puntos_pct`), así que agrupar, comparar o graficar por tipo es una sola instrucción (`groupby("tipo_salario")`) en vez de tener que repetirla a mano para cada columna del formato ancho.
- **Para agregaciones por grupos**, cualquier herramienta de agregación (`groupby`, `pivot_table`, o una librería de gráficos como Seaborn/Plotly) espera justo esta forma: una columna que diga **qué se mide** (`tipo_salario`, `componente`...) y otra con **el valor medido**. En ancho, cada agregación nueva obliga a escribir código específico por columna; en largo, el mismo código sirve para cualquier número de variables.
- **Para leer e interpretar**, el ancho es más cómodo cuando se quiere ver una fila = una persona con todo a la vista.

 El largo es mejor cuando la pregunta es "¿cómo se compara X entre categorías?", que es exactamente lo que se necesita antes de visualizar.


### Ejemplo de AGREGACIÓN: salario medio por tipo y por género, en una sola instrucción

In [312]:
print("Con formato largo (una sola agrupación, una sola columna de valor):")
display(largo_salarios.groupby(["tipo_salario", "Gènere"])["euros_mes"].mean().round(0).unstack())

Con formato largo (una sola agrupación, una sola columna de valor):


Gènere,Altre,Dona,Home,No consta
tipo_salario,,,,
Salario ajustado,1778.0,1580.0,1802.0,1673.0
Salario base,1655.0,1485.0,1694.0,1571.0


### El mismo resultado en formato ancho exige repetir la agregación una vez por columna



In [313]:
print("\nCon formato ancho (hay que repetir la operación para cada columna de salario):")
print("Salario base  :", df_enriquecido.groupby("Gènere")["Salari mensual"].mean().round(0).to_dict())
print("Salario ajustado:", df_enriquecido.groupby("Gènere")["salari_ajustat"].mean().round(0).to_dict())


Con formato ancho (hay que repetir la operación para cada columna de salario):
Salario base  : {'Altre': 1655.0, 'Dona': 1485.0, 'Home': 1694.0, 'No consta': 1571.0}
Salario ajustado: {'Altre': 1778.0, 'Dona': 1580.0, 'Home': 1802.0, 'No consta': 1673.0}


### Ejemplo de COMPARACIÓN entre magnitudes distintas (los 5 componentes del incremento) en una sola tabla/gráfico

In [314]:
print("Aportación media de cada componente al incremento total, por grupo profesional:")
display(largo_incremento.pivot_table(index="Grup Professional", columns="componente", values="puntos_pct", aggfunc="mean").round(2))

Aportación media de cada componente al incremento total, por grupo profesional:


componente,Coste de vida,Edad,Grupo profesional,Hijos,Transporte
Grup Professional,,,,,
Grup A,0.52,1.08,5.0,0.60,0.11
Grup B,0.58,1.10,3.5,0.54,0.10
Grup C,0.66,1.25,2.0,0.68,0.07
Grup D,0.42,0.99,8.0,0.71,0.04


# Ejercicio 3: Creación de variables derivadas (Feature Engineering)

Objetivo: generar nuevas variables derivadas a partir de la información existente y evaluar su calidad analítica. Aunque algunas variables de base (como la edad o el salario) han sido ya introducidas en ejercicios anteriores, en este nivel no se valora su creación técnica, sino la capacidad de transformarlas en indicadores más informativos, la detección de casos límite o incoherentes y la reflexión sobre la utilidad real de las variables generadas en un contexto analítico.

Ejemplos (elección guiada):

- » Índice de dependencia familiar: Salario mensual / (Número hijos + 1).
- » Índice de movilidad: Km anuales / Temperatura media.
- » Ratio salario / coste de vida.
- » Indicador de dependencia familiar: Salario / Número hijos.
- » Peso de responsabilidad familiar: Número hijos * Edad.
- » Indicador de etapa vital: 0 → sin cargas, 1 → etapa activa, 2 → alta dependencia.

Concretamente, tendrás que:

- » Crear al menos 3 variables derivadas.
- » Justificar:
  - Su utilidad analítica.
  - Las decisiones tomadas.
- » Detectar casos límite o incoherentes generados por las nuevas variables.

## 1. Variable derivada 1 — Índice de dependencia familiar

**Utilidad analítica:** el salario en bruto no dice cuánta gente vive de él. Este índice aproxima "salario disponible por miembro del hogar" (la propia persona + sus hijos), así que dos personas con el mismo sueldo pero distinto número de hijos dejan de parecer iguales: es más útil para comparar la situación económica real que el salario solo.

**Decisiones tomadas:**
- Se usa `Nombre_fills + 1` y no solo `Nombre_fills`, para **evitar la división por cero** en las 563 personas sin hijos
- Se usa el salario **base** (`Salari mensual`), no el ajustado, porque el incremento de la política (`salari_ajustat`) ya incluye un componente por hijos
- Se propaga el nulo de `Nombre_fills` (32 filas): no tiene sentido inventar un número de hijos para calcular este índice.

In [315]:
df_enriquecido["idx_dependencia_familiar"] = (
    df_enriquecido["Salari mensual"] / (df_enriquecido["Nombre_fills"] + 1)
).round(1)

print(df_enriquecido["idx_dependencia_familiar"].describe().round(1))
print("\nNulos:", df_enriquecido["idx_dependencia_familiar"].isna().sum(), "(mismos que Nombre_fills sin dato)")

count     953.0
mean     1145.9
std       675.8
min       143.6
25%       614.0
50%      1067.7
75%      1524.0
max      3672.0
Name: idx_dependencia_familiar, dtype: float64

Nulos: 32 (mismos que Nombre_fills sin dato)


In [316]:
print("Casos límite / incoherentes:")
extremos = df_enriquecido.nsmallest(5, "idx_dependencia_familiar")[["Nom", "Cognoms", "Salari mensual", "Nombre_fills", "idx_dependencia_familiar"]]
display(extremos)
print(f"\nPersonas con índice < 300 € por miembro del hogar: {(df_enriquecido['idx_dependencia_familiar'] < 300).sum()}")
print("→ son personas con 3-4 hijos Y un salario ya bajo de por sí: el índice combina ambos factores de riesgo a la vez,")

Casos límite / incoherentes:


,Nom,Cognoms,Salari mensual,Nombre_fills,idx_dependencia_familiar
342,Alejandro,Gómez López,718.0,4.0,143.6
259,Ilaria,Colombo,724.0,4.0,144.8
502,Marina,Puig Hernández,770.0,4.0,154.0
131,Marco,Rossi Ricci,771.0,4.0,154.2
559,Alessia,Mancini,775.0,4.0,155.0



Personas con índice < 300 € por miembro del hogar: 77
→ son personas con 3-4 hijos Y un salario ya bajo de por sí: el índice combina ambos factores de riesgo a la vez,


## 2. Variable derivada 2 — Ratio salario ajustado / coste de vida

**Utilidad analítica:** 1.800 € no valen lo mismo en una ciudad cara que en una barata. Dividir el salario por el índice de coste de vida da una medida de **poder adquisitivo relativo**, mucho más útil que el salario absoluto para comparar personas que viven en ciudades distintas.

**Decisiones tomadas:**
- Se usa el salario **ajustado** (`salari_ajustat`), porque es el que de verdad va a cobrar la persona tras aplicar la política; comparar el poder adquisitivo tiene más sentido con el salario final que con el de partida.
- El resultado es un número **sin unidad física** (euros ÷ índice, no euros ÷ euros), porque `Cost_vida_index` es un índice relativo del CSV, no un coste en euros.


In [317]:
df_enriquecido["ratio_salari_cost_vida"] = (
    df_enriquecido["salari_ajustat"] / df_enriquecido["Cost_vida_index"]
).round(2)

print(df_enriquecido["ratio_salari_cost_vida"].describe().round(2))
print("\nNulos:", df_enriquecido["ratio_salari_cost_vida"].isna().sum(),)

count    855.00
mean      23.85
std        9.69
min        9.38
25%       16.66
50%       21.54
75%       28.18
max       65.44
Name: ratio_salari_cost_vida, dtype: float64

Nulos: 130


In [318]:
print("Casos límite / incoherentes:")
top5 = df_enriquecido.nlargest(5, "ratio_salari_cost_vida")[["Nom", "Ciutat", "salari_ajustat", "Cost_vida_index", "ratio_salari_cost_vida"]]
bottom5 = df_enriquecido.nsmallest(5, "ratio_salari_cost_vida")[["Nom", "Ciutat", "salari_ajustat", "Cost_vida_index", "ratio_salari_cost_vida"]]
print("Mejor poder adquisitivo relativo:"); display(top5)
print("Peor poder adquisitivo relativo:"); display(bottom5)
print("→ el ratio más bajo junta salario pequeño CON ciudad cara a la vez (p. ej. Oslo, Noruega, Cost_vida_index=90):")


Casos límite / incoherentes:
Mejor poder adquisitivo relativo:


,Nom,Ciutat,salari_ajustat,Cost_vida_index,ratio_salari_cost_vida
12,Aina,Las Palmas de Gran Canaria,4057.56,62.0,65.44
693,Oriol,Murcia,3849.44,60.0,64.16
833,Víctor,Sevilla,3836.00,65.0,59.02
233,Laia,Alacant,3976.50,68.0,58.48
669,Alessandro,Napoli,3748.18,65.0,57.66


Peor poder adquisitivo relativo:


,Nom,Ciutat,salari_ajustat,Cost_vida_index,ratio_salari_cost_vida
574,Alessandro,Milano,750.07,80.0,9.38
663,Maja,Bergen,863.66,85.0,10.16
192,Julien,Marseille,772.54,72.0,10.73
342,Alejandro,Bilbao,775.44,72.0,10.77
559,Alessia,Roma,848.62,78.0,10.88


→ el ratio más bajo junta salario pequeño CON ciudad cara a la vez (p. ej. Oslo, Noruega, Cost_vida_index=90):


## 3. Variable derivada 3 — Peso de responsabilidad familiar

**Utilidad analítica:** no es lo mismo tener 2 hijos a los 30 años (con toda la etapa de crianza por delante) que a los 55 (posiblemente ya independizados o a punto de estarlo). Multiplicar hijos × edad da una idea aproximada de **carga acumulada** más que de carga futura: crece tanto si hay más hijos como si la persona es mayor, dos señales que por separado no capturan lo mismo.

**Decisiones tomadas:**
- Es una variable **deliberadamente simple** (un producto, no una fórmula demográfica real) — se advierte de que es un indicador aproximado, no una medida validada, y de que su utilidad es puramente comparativa dentro de este dataset.

- Se dejan como nulos los casos sin hijos conocidos O sin edad conocida, en vez de forzar un 0 o una media: multiplicar por un valor inventado falsearía la magnitud, no solo su signo.

In [319]:
df_enriquecido["peso_responsabilidad_familiar"] = df_enriquecido["Nombre_fills"] * df_enriquecido["edat"]
print(df_enriquecido["peso_responsabilidad_familiar"].describe().round(1))
print("\nNulos:", df_enriquecido["peso_responsabilidad_familiar"].isna().sum(),
      "= hijos desconocidos (32) + edad desconocida (77) - solapamiento, sin doble contar")

count    877.0
mean      39.0
std       61.1
min        0.0
25%        0.0
50%        0.0
75%       60.0
max      268.0
Name: peso_responsabilidad_familiar, dtype: float64

Nulos: 108 = hijos desconocidos (32) + edad desconocida (77) - solapamiento, sin doble contar


In [320]:
print("Casos límite / incoherentes:")
top5 = df_enriquecido.nlargest(5, "peso_responsabilidad_familiar")[["Nom", "edat", "Nombre_fills", "peso_responsabilidad_familiar"]]
display(top5)
print(f"\nPersonas con 4 hijos Y en la franja 60+ a la vez: {((df_enriquecido['Nombre_fills']==4) & (df_enriquecido['franja_edat']=='60+')).sum()}")
print("\n→ estos son los valores más altos del indicador: máxima carga familiar declarada Y edad avanzada a la vez,")
print("  un perfil que convendría revisar a mano si este indicador se usara para algo operativo (p. ej. beneficios sociales).")

Casos límite / incoherentes:


,Nom,edat,Nombre_fills,peso_responsabilidad_familiar
127,Léa,67.0,4.0,268.0
372,Pol,67.0,4.0,268.0
534,Lucía,67.0,4.0,268.0
378,Henrik,66.0,4.0,264.0
566,Carmen,66.0,4.0,264.0



Personas con 4 hijos Y en la franja 60+ a la vez: 12

→ estos son los valores más altos del indicador: máxima carga familiar declarada Y edad avanzada a la vez,
  un perfil que convendría revisar a mano si este indicador se usara para algo operativo (p. ej. beneficios sociales).


## 4. Variable derivada 4 — Indicador de etapa vital

**Utilidad analítica:** reduce dos variables numéricas (edad, nº de hijos) a una sola categoría hace que sea mucho más fácil de cruzar con otras variables (grupo, género, ciudad) que manejar edad y nº de hijos por separado.

**Decisiones tomadas:**
- Se combinan **dos criterios de "alta dependencia"**: tener 3 o más hijos, o tener hijos y estar en la franja 60+.
- Cuando falta la franja de edad pero la persona tiene 1-2 hijos, **no se asume "etapa activa"**: podría ser una persona de 60+ años con esos mismos hijos, que cambiaría la categoría a "alta dependencia". Ante la duda, se deja en nulo en vez de arriesgar una clasificación incorrecta.


In [321]:
def etapa_vital(fila):
    """0 = sin cargas | 1 = etapa activa | 2 = alta dependencia | NaN = no se puede determinar."""
    if pd.isna(fila["te_fills"]):
        return np.nan
    if fila["te_fills"] == False:
        return 0
    if fila["Nombre_fills"] >= 3:
        return 2
    if fila["franja_edat"] == "60+":
        return 2
    if pd.isna(fila["franja_edat"]):
        return np.nan
    return 1

df_enriquecido["etapa_vital"] = df_enriquecido.apply(etapa_vital, axis=1)

ETIQUETAS = {0: "Sin cargas", 1: "Etapa activa", 2: "Alta dependencia"}
df_enriquecido["etapa_vital_txt"] = df_enriquecido["etapa_vital"].map(ETIQUETAS)

print(df_enriquecido["etapa_vital_txt"].value_counts(dropna=False))

etapa_vital_txt
Sin cargas          563
Etapa activa        186
Alta dependencia    184
NaN                  52
Name: count, dtype: int64


In [322]:
print("Casos límite:")
print(f"  Sin franja de edad Y con 1-2 hijos (quedan sin clasificar por precaución): "
      f"{(df_enriquecido['etapa_vital'].isna() & df_enriquecido['te_fills'].fillna(False) & (df_enriquecido['Nombre_fills'] < 3)).sum()}")
print(f"  Alta dependencia por volumen de hijos (≥3), sin llegar a 60+: "
      f"{((df_enriquecido['etapa_vital']==2) & (df_enriquecido['franja_edat']!='60+')).sum()}")
print(f"  Alta dependencia por edad (60+), con menos de 3 hijos: "
      f"{((df_enriquecido['etapa_vital']==2) & (df_enriquecido['Nombre_fills']<3)).sum()}")
print("→ confirma que la categoría 'alta dependencia' recoge dos perfiles distintos (muchos hijos / hijos + edad),")
print("  algo a tener en cuenta si se usa esta variable para decisiones: no todo '2' significa lo mismo.")

Casos límite:
  Sin franja de edad Y con 1-2 hijos (quedan sin clasificar por precaución): 20
  Alta dependencia por volumen de hijos (≥3), sin llegar a 60+: 114
  Alta dependencia por edad (60+), con menos de 3 hijos: 43
→ confirma que la categoría 'alta dependencia' recoge dos perfiles distintos (muchos hijos / hijos + edad),
  algo a tener en cuenta si se usa esta variable para decisiones: no todo '2' significa lo mismo.


## 5. Resumen de las variables creadas

In [323]:
resumen = pd.DataFrame({
    "variable": ["idx_dependencia_familiar", "ratio_salari_cost_vida", "peso_responsabilidad_familiar", "etapa_vital"],
    "fórmula": ["Salario / (hijos+1)", "Salario ajustado / índice coste de vida", "hijos × edad", "regla combinada hijos+edad"],
    "nulos": [df_enriquecido[c].isna().sum() for c in
              ["idx_dependencia_familiar", "ratio_salari_cost_vida", "peso_responsabilidad_familiar", "etapa_vital"]],
    "motivo del nulo": ["hijos desconocidos", "ciudad sin contexto", "hijos y/o edad desconocidos", "hijos y/o edad desconocidos"],
})
display(resumen)

df_features = df_enriquecido.reset_index(drop=True)
print("\nDataset con variables derivadas:", df_features.shape)

,variable,fórmula,nulos,motivo del nulo
0,idx_dependencia_familiar,Salario / (hijos+1),32,hijos desconocidos
1,ratio_salari_cost_vida,Salario ajustado / índice coste de vida,130,ciudad sin contexto
2,peso_responsabilidad_familiar,hijos × edad,108,hijos y/o edad desconocidos
3,etapa_vital,regla combinada hijos+edad,52,hijos y/o edad desconocidos



Dataset con variables derivadas: (985, 42)


#Ejercicio 4: Primeros análisis descriptivos

- » Construir tablas resumen descriptivas.
- » Realizar comparaciones básicas entre grupos relevantes.

Concretamente, tendrás que:

- » Construir tablas resumen descriptivas de salarios como una tabla resumen que permita ver el sueldo medio, medio, mínimo y máximo por Género. Ordena la tabla en función del sueldo medio.
- » Crear una tabla resumen con el salario medio por género (filas) y país de origen (columnas). Añade las medias a los márgenes de la mesa. Aplica formato condicional a la mesa para ver en un color más intenso los valores más elevados.

## 1. ¿Qué nivel de agrupación es el más adecuado?

(`Grup Professional`):

- Es unha variable sin ningún nulo así que agrupar por ella no deja ningún registro "huérfano" sin archivo al que pertenecer.
- Es la variable que más influye en la política salarial (`base_increment_by_group`) y en el salario en sí, así que separar por grupo profesional es mas útil

In [324]:
grupos = sorted(df_features["Grup Professional"].unique())
print("Grupos profesionales a exportar:", grupos)
print(df_features["Grup Professional"].value_counts().sort_index())

Grupos profesionales a exportar: ['Grup A', 'Grup B', 'Grup C', 'Grup D']
Grup Professional
Grup A    456
Grup B    350
Grup C    139
Grup D     40
Name: count, dtype: int64


## 2. Separar datos detallados de datos agregados

Se generan dos niveles de salida, pensados para usos distintos:
- **Detalle** (1 archivo por grupo): una fila por persona, con todas las columnas preparadas/enriquecidas
- **Agregado** (1 archivo único): una fila por grupo, solo con los estadísticos clave

Mezclar ambos niveles en el mismo archivo obligaría a quien lo abra a filtrar o a hacer el `groupby` él mismo cada vez; separarlos deja cada archivo listo para su uso directo.

### 2.1 Qué columnas exportar en el detalle

In [325]:
columnas_excluir = ["idx_origen"]
columnas_detalle = [c for c in df_features.columns if c not in columnas_excluir]
print(f"Columnas en el detalle: {len(columnas_detalle)} de {len(df_features.columns)}")

Columnas en el detalle: 41 de 42


## 3. Exportar un archivo por Grupo Profesional (con un bucle)

**Decisiones tomadas:**
- Se usa `.csv` (no `.xlsx`) para el detalle: son 985 filas en total repartidas en 4 archivos, un formato más ligero y universal para datos tabulares sin formato visual.
- `encoding="utf-8-sig"` en vez de `"utf-8"` a secas: así Excel reconoce automáticamente los acentos y la `ñ`/`ç` sin que la persona que lo abra tenga que configurar nada

In [326]:
carpeta_salida = "salidas_grupo"
os.makedirs(carpeta_salida, exist_ok=True)

archivos_generados = []
for grupo in grupos:
    datos_grupo = df_features.loc[df_features["Grup Professional"] == grupo, columnas_detalle]
    sufijo = grupo.replace("Grup ", "")
    nombre_archivo = f"{carpeta_salida}/datos_Grup{sufijo}.csv"
    datos_grupo.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")   #
    archivos_generados.append(nombre_archivo)
    print(f"{nombre_archivo:35s} → {len(datos_grupo):4d} filas, {len(columnas_detalle)} columnas")

print(f"\nTotal de archivos generados: {len(archivos_generados)}")

salidas_grupo/datos_GrupA.csv       →  456 filas, 41 columnas
salidas_grupo/datos_GrupB.csv       →  350 filas, 41 columnas
salidas_grupo/datos_GrupC.csv       →  139 filas, 41 columnas
salidas_grupo/datos_GrupD.csv       →   40 filas, 41 columnas

Total de archivos generados: 4


## 4. Quinto archivo: resumen agregado por Grupo Profesional

**Qué información se incluye, y por qué:**
- `n_trabajadores`: el tamaño de cada grupo — imprescindible para no leer el salario o la edad media de un grupo pequeño como si tuviera el mismo peso que uno grande.
- `salario_medio`: se usa el salario **base**, no el ajustado, porque el resumen describe la situación de partida de cada grupo, no el resultado de aplicar la política salarial (que ya se detalla por separado en `increment_aplicat_pct` dentro de cada archivo de detalle).
- `edad_media`: calculada solo con las personas con fecha de nacimiento válida; se muestra cuántas personas entran en cada media, para que quien lo use sepa que no es el 100 % de la plantilla.

In [327]:
resumen_grupos = (
    df_features.groupby("Grup Professional")
    .agg(
        n_trabajadores=("DNI", "count"),
        salario_medio=("Salari mensual", "mean"),
        edad_media=("edat", "mean"),
    )
    .round(1)
    .reset_index()
)

print("Nota: la edad media se calcula solo con las personas que tienen una fecha de nacimiento válida.")
print(df_features.groupby("Grup Professional")["edat"].apply(lambda s: s.notna().sum()).rename("con edad válida"))

display(resumen_grupos)

resumen_grupos.to_csv("salidas_grupo/resumen_grupos_profesionales.csv", index=False, encoding="utf-8-sig")
print("\nArchivo exportado: salidas_grupo/resumen_grupos_profesionales.csv")

Nota: la edad media se calcula solo con las personas que tienen una fecha de nacimiento válida.
Grup Professional
Grup A    420
Grup B    320
Grup C    130
Grup D     38
Name: con edad válida, dtype: int64


,Grup Professional,n_trabajadores,salario_medio,edad_media
0,Grup A,456,1117.1,43.6
1,Grup B,350,1649.9,44.0
2,Grup C,139,2359.5,45.6
3,Grup D,40,3166.6,40.5



Archivo exportado: salidas_grupo/resumen_grupos_profesionales.csv


## 6. Resumen de todo lo exportado

In [328]:
print("Archivos generados en 'salidas_grupo/':\n")
for ruta in sorted(pathlib.Path("salidas_grupo").iterdir()):
    print(f"  {ruta.name:45s} {ruta.stat().st_size / 1024:6.1f} KB")

Archivos generados en 'salidas_grupo/':

  datos_GrupA.csv                                107.2 KB
  datos_GrupB.csv                                 82.5 KB
  datos_GrupC.csv                                 33.2 KB
  datos_GrupD.csv                                 10.1 KB
  resumen_grupos_profesionales.csv                 0.1 KB
